## Tahap 0.1 — Lingkungan

**Dasar naskah:** Bab III (lingkungan komputasi: Google Colab). GPU memakai **A100**, menggantikan T4 yang tertulis di Bab III (keputusan pengguna 21 Oktober 2026; naskah perlu disesuaikan). `CLAUDE.md` Bagian 3.1, 3.3, 3.4, dan 10.

**Package:** `os`, `sys`, `subprocess`, `importlib`, `warnings`, `pathlib`, `json`, `time`, `platform` (bawaan Python); `numpy`, `pandas`, `torch`, `transformers`, `matplotlib` (di-*import*); `scikit-learn`, `scipy`, `openpyxl`, `pyarrow` (dipasang bila belum ada, versinya dicatat); `google.colab.drive` (*mount* Google Drive).

Jalankan sel 1/2 lalu sel 2/2 **paling awal** setiap kali *runtime* dimulai atau di-*restart*, sebelum Tahap 0.2.

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 1/2: variabel lingkungan, folder kerja, pemasangan package) ══
# Package: os, sys, subprocess, importlib, warnings, pathlib (bawaan Python); google.colab.drive
import os, sys, subprocess, importlib.util, warnings
from pathlib import Path
from google.colab import drive

# 1) Variabel lingkungan, diatur SEBELUM torch/transformers di-import (3.3–3.4)
_cuda_dulu = "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized()
_cublas_dulu = os.environ.get("CUBLAS_WORKSPACE_CONFIG")
os.environ.update({
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",                                          # determinisme cuBLAS
    "TOKENIZERS_PARALLELISM": "false",                                             # senyap: tokenizer
    "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1",   # senyap: transformers
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "HF_DATASETS_DISABLE_PROGRESS_BARS": "1", # senyap: HF Hub, datasets
    "HF_HUB_VERBOSITY": "error",                                                   # senyap: log HF Hub (token)
    "TQDM_DISABLE": "1",                                                           # senyap: semua tqdm
    "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3",                                    # jangan muat TensorFlow
})
warnings.filterwarnings("ignore")

# 2) Google Drive dan struktur folder (3.1); folder versi lama tidak disentuh
if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except ValueError as e:
        raise RuntimeError("Mount gagal: pilih akun yang menyimpan folder skripsi, centang SEMUA izin, ulangi sel.") from e
AKAR = Path("/content/drive/MyDrive/skripsi_wondr_v3")
AKAR.mkdir(exist_ok=True)                         # gagal jika MyDrive belum ter-mount
for sub in ["data", "lib", "hasil", "anotasi", "bab4/tabel", "bab4/gambar", "log"]:
    (AKAR / sub).mkdir(parents=True, exist_ok=True)
(AKAR / "lib" / "__init__.py").touch()            # agar "from lib.wondr_teks import ..." jalan
os.chdir(AKAR)                                     # agar %%writefile cukup memakai lib/...
if str(AKAR) not in sys.path:
    sys.path.insert(0, str(AKAR))

# 3) Pasang hanya package yang belum ada (di-import di sel 2/2)
PAKET = {"numpy": "numpy", "pandas": "pandas", "torch": "torch", "transformers": "transformers",
         "sklearn": "scikit-learn", "scipy": "scipy", "matplotlib": "matplotlib",
         "openpyxl": "openpyxl", "pyarrow": "pyarrow"}            # nama modul -> nama pip
_dipasang = [pip for mod, pip in PAKET.items() if importlib.util.find_spec(mod) is None]
if _dipasang:
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_dipasang], capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + _r.stderr[-2000:])

In [ ]:
# ══ Tahap 0.1 — Lingkungan (sel 2/2: import, GPU, dan log versi) ══
# Package: importlib, json, time, platform (bawaan Python); numpy, pandas, torch, transformers,
#          matplotlib (di-import); scikit-learn, scipy, openpyxl, pyarrow (versinya dicatat)
import importlib.metadata, json, time, platform
t0 = time.perf_counter()
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, transformers
from huggingface_hub.utils import logging as hf_logging

# 1) Matikan kebisingan yang tersisa setelah import
transformers.logging.set_verbosity_error()        # sembunyikan "Some weights ... newly initialized"
transformers.logging.disable_progress_bar()       # progress bar unduhan model
hf_logging.set_verbosity_error()                  # sembunyikan peringatan HF Hub "unauthenticated requests"
plt.rcParams.update({"savefig.dpi": 300, "savefig.bbox": "tight"})   # PNG 300 dpi (3.1)

# 2) GPU dan versi -> log/versi_package.json (acuan = sesi A100 pertama)
GPU_PENELITIAN = "A100"                            # Keputusan pengguna 21-10-2026, menggantikan T4
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3 if gpu != "tidak ada" else 0.0
sesi = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "gpu": gpu,
        "gpu_memori_gb": round(mem, 2), "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
        "package": {p: importlib.metadata.version(p) for p in [*PAKET.values(), "tokenizers", "huggingface-hub"]}}
log = AKAR / "log" / "versi_package.json"
lama = json.loads(log.read_text()) if log.exists() else {}
pertama = lama["pertama"] if GPU_PENELITIAN in lama.get("pertama", {}).get("gpu", "") else sesi
log.write_text(json.dumps({"pertama": pertama, "terbaru": sesi}, indent=2))
beda = [k for k in ("torch", "transformers", "tokenizers") if pertama["package"][k] != sesi["package"][k]]
beda += ["GPU"] if pertama["gpu"] != gpu else []

# 3) Ringkasan + cek (judul() dan cek() baru dibuat di Tahap 0.2)
xlsx, koma = AKAR / "data" / "data_wonder_bybni_3.xlsx", lambda x: f"{x:.1f}".replace(".", ",")
cublas_ok = not _cuda_dulu or _cublas_dulu == ":4096:8"
print(f"══════════ TAHAP 0.1 — LINGKUNGAN ══════════\nMasukan : data/{xlsx.name}\nKeluaran: log/{log.name}")
print(f"{'─' * 48}\nFolder kerja : {AKAR}\nPython       : {sesi['python']}\n"
      f"GPU          : {gpu} ({koma(mem)} GB) · CUDA {sesi['cuda']} · cuDNN {sesi['cudnn']}\n\n{'Package':<17}Versi")
print("\n".join(f"{k:<17}{v}{'  (baru dipasang)' * (k in _dipasang)}" for k, v in sesi["package"].items()))
print(f"{'─' * 48}\nCek otomatis:")
for nama, ok, aktual, target in [
        ("berkas data ada", xlsx.exists(), "ya" if xlsx.exists() else "tidak", "ya"),
        ("GPU", GPU_PENELITIAN in gpu, gpu, f"memuat {GPU_PENELITIAN}"),
        ("CUBLAS_WORKSPACE_CONFIG diatur sebelum CUDA aktif", cublas_ok, "ya" if cublas_ok else "tidak, restart runtime", "ya"),
        ("versi torch/transformers/tokenizers/GPU", not beda, "berubah: " + ", ".join(beda) if beda else "sama", "sama")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")
print(f"Waktu: {koma(time.perf_counter() - t0)} detik")

## Tahap 0.2 — Konfigurasi dan fungsi bantu

**Dasar naskah:** `CLAUDE.md` Bagian 4 (`CFG`, dengan asal setiap nilai di komentar) dan Bagian 6 (`TARGET`); Bagian 3.4 (`atur_seed()`). Keputusan teknis KT-1 sampai KT-4 disetujui pengguna pada 21 Oktober 2026 (lihat `CLAUDE.md` Bagian 10).

**Package:** `json`, `math`, `os`, `random`, `time`, `pathlib`, `importlib` (bawaan Python); `numpy`; `torch` dan `transformers` (di `atur_seed()`); `matplotlib` (di `simpan_gambar()`); `pandas` (lewat `DataFrame.to_csv` di `simpan_tabel()`).

Sel 1/2 menulis `lib/bantu.py`, yang isinya fungsi saja. Sel 2/2 membuat `PATH`, `CFG`, dan `TARGET`. Setelah *runtime* di-*restart*, jalankan Tahap 0.1 (2 sel) lalu Tahap 0.2 (2 sel).

In [ ]:
%%writefile lib/bantu.py
# lib/bantu.py -- fungsi bantu untuk semua subtahap (Tahap 0.2; KT-4 disetujui 21-10-2026).
# Isinya fungsi saja; PATH, CFG, dan TARGET dibuat di sel 2/2 Tahap 0.2.
import json, math, os, random, time
from pathlib import Path
import numpy as np

__all__ = ["fmt_id", "rel", "mulai", "judul", "garis", "cek", "selesai",
           "simpan_tabel", "simpan_gambar", "simpan_json", "baca_json", "atur_seed"]
_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_STATUS = {"t0": None, "cek_mulai": False}
_LEBAR = 48                                           # lebar garis pemisah blok ringkasan


def _angka(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def fmt_id(x, d=None):
    """Format angka Indonesia: 7204 -> '7.204'; 0.836 -> '0,8360'. Selain angka -> str(x)."""
    if not _angka(x):
        return str(x)
    if d is None:
        d = 0 if isinstance(x, (int, np.integer)) else 4
    return f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def _tampil(v, d=None):
    """Nilai untuk ringkasan: bilangan bulat tanpa desimal, pecahan d desimal, daftar dipisah ' · '."""
    if isinstance(v, np.ndarray):
        v = v.tolist()
    if isinstance(v, dict):
        return ", ".join(f"{k}: {_tampil(x, d)}" for k, x in v.items())
    if isinstance(v, (list, tuple)):
        return " · ".join(f"[{_tampil(x, d)}]" if isinstance(x, (list, tuple)) else _tampil(x, d) for x in v)
    return fmt_id(v) if isinstance(v, (int, np.integer)) else fmt_id(v, d)


def rel(p):
    """Path relatif terhadap folder kerja, untuk ditampilkan."""
    try:
        return str(Path(p).resolve().relative_to(_AKAR))
    except ValueError:
        return str(p)


def garis():
    print("─" * _LEBAR)


def mulai():
    """Dipanggil di awal sel: mulai menghitung waktu."""
    _STATUS["t0"] = time.perf_counter()


def judul(no, nama, masukan=None, keluaran=None):
    """Kepala blok ringkasan; masukan/keluaran berupa str atau list of str."""
    _STATUS["cek_mulai"] = False
    print(f"{'═' * 10} TAHAP {no} — {nama.upper()} {'═' * 10}")
    for label, isi in (("Masukan ", masukan), ("Keluaran", keluaran)):
        if isi:
            isi = [isi] if isinstance(isi, str) else list(isi)
            print(f"{label}: " + f"\n{' ' * 10}".join(isi))
    garis()


def _sama(a, t, tol):
    a = a.tolist() if isinstance(a, np.ndarray) else a
    t = t.tolist() if isinstance(t, np.ndarray) else t
    if _angka(a) and _angka(t):
        return abs(float(a) - float(t)) <= tol
    if isinstance(a, dict) and isinstance(t, dict):
        return a.keys() == t.keys() and all(_sama(a[k], t[k], tol) for k in t)
    if isinstance(a, (list, tuple)) and isinstance(t, (list, tuple)):
        return len(a) == len(t) and all(_sama(x, y, tol) for x, y in zip(a, t))
    return bool(a == t)


def cek(nama, aktual, target, toleransi=0, cara="=", d=None):
    """Cetak ✅/⚠️ beserta nilai aktual dan target, tanpa assert. cara: '=', '>', '>=', '<', '<='."""
    if not _STATUS["cek_mulai"]:
        garis()
        print("Cek otomatis:")
        _STATUS["cek_mulai"] = True
    if d is None and toleransi > 0:
        d = max(0, round(-math.log10(2 * toleransi)))  # 0,05 -> 1 desimal; 5e-5 -> 4 desimal
    if cara == "=":
        ok = _sama(aktual, target, toleransi)
    else:
        a, t = float(aktual), float(target)
        ok = {">": a > t, ">=": a >= t, "<": a < t, "<=": a <= t}[cara]
    tanda = "" if cara == "=" else cara + " "
    print(f"  {'✅' if ok else '⚠️'} {nama} = {_tampil(aktual, d)} (target {tanda}{_tampil(target, d)})")
    return bool(ok)


def selesai():
    """Baris terakhir blok ringkasan: lama sel berjalan sejak mulai()."""
    dt = time.perf_counter() - _STATUS["t0"] if _STATUS["t0"] else 0.0
    teks = (f"{fmt_id(dt, 1)} detik" if dt < 120 else
            f"{fmt_id(dt / 60, 1)} menit" if dt < 7200 else f"{fmt_id(dt / 3600, 2)} jam")
    print(f"Waktu: {teks}")


def simpan_tabel(df, nama, index=False):
    """CSV untuk Bab IV (angka mentah) -> bab4/tabel/<nama>.csv; mengembalikan path-nya."""
    p = _AKAR / "bab4" / "tabel" / f"{nama}.csv"
    df.to_csv(p, index=index)
    return p


def simpan_gambar(fig, nama):
    """PNG 300 dpi -> bab4/gambar/<nama>.png, lalu ditampilkan satu kali."""
    import matplotlib.pyplot as plt
    p = _AKAR / "bab4" / "gambar" / f"{nama}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return p


def _ke_json(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, (np.ndarray, Path)):
        return o.tolist() if isinstance(o, np.ndarray) else str(o)
    raise TypeError(f"tidak bisa disimpan ke JSON: {type(o).__name__}")


def simpan_json(obj, path):
    """Tulis JSON secara atomik (berkas .tmp lalu diganti), aman bila Colab terputus."""
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=_ke_json), encoding="utf-8")
    os.replace(tmp, path)
    return path


def baca_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atur_seed(s):
    """Satu seed untuk semua sumber keacakan pelatihan (3.4), ditambah KT-2 dan KT-3 (21-10-2026)."""
    import torch, transformers
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    transformers.set_seed(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)         # KT-2: operasi GPU tak-deterministik -> error
    torch.backends.cuda.matmul.allow_tf32 = False    # KT-3: FP32 penuh; API lama ikut mengatur API baru,
    torch.backends.cudnn.allow_tf32 = False          #       API baru saja tidak mematikan TF32 konv/RNN cuDNN


def uji_bantu(s):
    """Uji Tahap 0.2: atur_seed(s) dua kali -> angka acak sama; status mode deterministik dan TF32."""
    import torch
    atur_seed(s); a = np.random.rand(3), torch.rand(3)
    atur_seed(s); b = np.random.rand(3), torch.rand(3)
    return {"ulang_sama": all(np.array_equal(x, y) for x, y in zip(a, b)),
            "deterministik": torch.are_deterministic_algorithms_enabled(),
            "tf32": [torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32]}

In [ ]:
# ══ Tahap 0.2 — Konfigurasi dan fungsi bantu (sel 2/2: PATH, CFG, TARGET) ══
# Package: importlib (bawaan Python); lib.bantu (fungsi bantu dari sel 1/2)
import importlib, lib.bantu
importlib.reload(lib.bantu)                         # pakai isi lib/bantu.py yang terbaru
from lib.bantu import *
mulai()

# 1) Lokasi folder dan berkas (3.1)
PATH = {"akar": AKAR, **{k: AKAR / k for k in ("data", "lib", "hasil", "anotasi", "log")},
        "tabel": AKAR / "bab4/tabel", "gambar": AKAR / "bab4/gambar", "xlsx": AKAR / "data/data_wonder_bybni_3.xlsx"}

# 2) Konfigurasi tetap (Bagian 4) -- jangan diubah
CFG = {
    "gpu": "A100",                                                  # Keputusan pengguna 21-10-2026
    "checkpoint": "indobenchmark/indobert-base-p1",                 # Bab III tahap 7; Wilie dkk. (2020)
    "tau": 4, "kelas": {0: "Non-Positif", 1: "Positif"},            # pers. (2.1)-(2.2): rating >= 4 -> 1
    "proporsi": {"latih": 0.70, "validasi": 0.15, "uji": 0.15},     # Bab III tahap 6
    "seed_bagi": 42, "kelompok_panjang": ("1-2", "3-10", ">10"),    # Bab III tahap 6
    "q": 100, "b": 32,                                              # pers. (2.9): L_max dari data latih saja
    "cl_lipatan": 5, "cl_seed_lipatan": 42, "cl_seed_latih": 42,    # Bab III tahap 8; Keputusan teknis
    "seed_anotasi": 2026,                                           # Keputusan: urutan lembar anotasi (tahap 9)
    "B": 16, "E": 4, "p": 2, "v": 1.0,                              # Tabel 3.1: batch, epoch, patience, norma
    "adam_betas": (0.9, 0.999), "adam_eps": 1e-8,                   # pers. (2.18); bawaan PyTorch
    "default": {"lr": 2e-5, "wr": 0.1, "wd": 0.01},                 # Tabel 3.1; Devlin dkk.; Sun dkk.
    "ruang": {"lr": (1e-5, 5e-5), "wr": (0.0, 0.2), "wd": (0.0, 0.3)},   # Tabel 3.1; pers. (2.27); lr skala log
    "seed_fitness": (42, 43),                                       # Bab III tahap 10; pers. (2.25)
    "rs_M": 50, "rs_seed": 2024,                                    # Bab III tahap 11
    "pso": {"s": 10, "G_max": 4, "omega": 0.729, "c1": 1.49445, "c2": 1.49445, "V_max": 1.0, "seed": 7},  # Tabel 3.2
    "seed_akhir": (42, 43, 44),                                     # Bab III tahap 13
    "R": 10_000, "seed_bootstrap": 42,                              # Bab III tahap 15; pers. (2.41)-(2.43)
    "alpha": 0.05, "m_uji": 3,                                      # pers. (2.44)-(2.45): alpha' = 0,05/3
    "kembaran_dekat": (339, 595, 2854, 3946, 4128),                 # Data (temuan pembagian)
}

# 3) Angka target (Bagian 6; nomor tahap mengikuti Bab III, lihat CLAUDE.md Bagian 10)
TARGET = {
    "1.1": {"baris": 7204, "kolom": ["nama", "rating", "ulasan", "tanggal", "likes", "balasan_dev", "versi"],
            "tanggal": ("2026-07-19", "2026-09-11"), "versi": ["1.6.0"],
            "per_bulan": {"2026-07": 1747, "2026-08": 3899, "2026-09": 1558}},
    "1.2": {"kosong": {"balasan_dev": 4834}, "dibalas_pct": {"1-4": 95.0, "5": 1.0}, "likes_nol_pct": 95.1,
            "spearman": -0.2483, "r3_panjang": (237, 366), "median_kata": [15, 15, 11, 3, 2], "emoji_simbol": 404,
            "huruf_ulang": (227, 3.2), "baca_ulang": 634, "duplikat_pct": 34.8, "label_ganda": (33, 1968),
            "nama_unik": 7153},
    "2": {"baris": 7204, "kolom": ["review_id", "ulasan", "rating", "tanggal", "versi"]},
    "3": {"per_rating": {1: 1305, 2: 283, 3: 366, 4: 492, 5: 4758}, "positif": (5250, 72.88),
          "non_positif": (1954, 27.12)},
    "4": {"contoh_bab2": ["baguss bgt! tp kadang2 error, nunggu lamaa", "mantap"],
          "berubah": [55, 404, 1518, 227, 634, 403], "kosong": 63, "kosong_rating": {5: 57, 4: 6}, "hasil": 7141},
    "5": {"kunci": 4372, "kunci_seri": 8, "ulasan_seri": 16, "dokumen": 4364, "dibuang": (2777, 2688, 73, 16),
          "positif_pct": (72.6, 57.3)},
    "6.1": {"grup": 4293, "latih": (3051, 1303, 1748), "validasi": (657, 282, 375), "uji": (656, 280, 376)},
    "6.2": {"bocor": 0, "kembaran_di_uji": 5, "non_positif_pct": (42.7, 42.9, 42.7), "p_label": 0.9945,
            "p_panjang": 0.9998, "panjang_pct": {"latih": (14.0, 47.1, 38.9), "validasi": (14.3, 46.9, 38.8),
                                                 "uji": (14.0, 47.1, 38.9)}},
    "7": {"L_max": 128, "terpotong": {"latih": 0, "validasi": 0, "uji": 1}, "terpotong_id": [2741]},
    "8.1": {"T": 764, "T_w": 76},                                   # n = 3.051, rho = 0,1
    "8.3": {"t0": 0.6167, "t1": 0.6667, "C": [[2, 1], [1, 2]], "kandidat": [3, 6]},    # contoh Tabel 2.9
    "9.2": {"kappa_contoh": 0.70, "toleransi": 1e-12},              # contoh Bab II 2.3.5
    "10": {"T": 764, "T_w": 76}, "11": {"kandidat": 50, "pelatihan": 100},
    "12.1": {"iter1": (5.6237, 3.6355), "iter2": (2.6718, 3.3698), "gbest_iter2": 9.8923,
             "toleransi": 5e-4, "evaluasi": 50},                    # contoh Tabel 2.17
    "13": {"pelatihan": 9, "toleransi": 1e-4}, "14.1": {"n_uji": 656},
    "15": {"f1_contoh": 0.6703, "persentil": (0.8333, 99.1667), "toleransi": 1e-12},   # contoh Tabel 2.19
}

# 4) Uji singkat fungsi bantu, lalu ringkasan
u = lib.bantu.uji_bantu(CFG["seed_fitness"][0])
judul("0.2", "Konfigurasi dan fungsi bantu", masukan="lib/bantu.py", keluaran="PATH, CFG, TARGET (di memori)")
print(f"CFG {len(CFG)} entri · TARGET {len(TARGET)} subtahap · PATH {len(PATH)} lokasi · "
      f"fungsi bantu {len(lib.bantu.__all__)}")
cek("fmt_id(7204) · fmt_id(0.836)", [fmt_id(7204), fmt_id(0.836)], ["7.204", "0,8360"])
cek("atur_seed() dua kali, angka acak sama", "ya" if u["ulang_sama"] else "tidak", "ya")
cek("mode deterministik PyTorch (KT-2)", "aktif" if u["deterministik"] else "mati", "aktif")
cek("TF32 matmul · cuDNN (KT-3)", ["nyala" if x else "mati" for x in u["tf32"]], ["mati", "mati"])
selesai()

## Tahap 1.1 — Muat data mentah

**Dasar naskah:** Bab III tahap 1 (Pengumpulan Data) dan subbab 3.1: ulasan 19 Juli–11 September 2026, versi aplikasi 1.6.0, disimpan apa adanya dalam format Excel dan tidak diubah. `review_id` dibuat di Tahap 2 sesuai Bab III tahap 2 (`CLAUDE.md` Bagian 10, keputusan no. 7).

**Package:** `pandas` (baca .xlsx, olah tabel), `openpyxl` (mesin baca .xlsx), `pyarrow` (mesin simpan .parquet); `lib.bantu` (fungsi bantu Tahap 0.2).

In [ ]:
# ══ Tahap 1.1 — Muat data mentah ══
# Package: pandas (baca .xlsx, olah tabel), openpyxl (mesin baca .xlsx), pyarrow (mesin simpan .parquet)
import pandas as pd
mulai()
T = TARGET["1.1"]
BULAN = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
         "Agustus", "September", "Oktober", "November", "Desember"]
tgl_id = lambda t: f"{t.day} {BULAN[t.month - 1]} {t.year}"       # 2026-07-19 -> "19 Juli 2026"

# 1) Baca data mentah (.xlsx)
mentah = pd.read_excel(PATH["xlsx"], engine="openpyxl")

# 2) Buang kolom indeks bawaan Excel (Unnamed*); isi tidak diubah, review_id dibuat di Tahap 2
buang = [c for c in mentah.columns if str(c).startswith("Unnamed")]
df = mentah.drop(columns=buang)

# 3) Simpan, lalu pastikan berkas yang dibaca ulang identik
keluar = PATH["hasil"] / "tahap01_mentah.parquet"
df.to_parquet(keluar, index=False)
identik = df.equals(pd.read_parquet(keluar))

# 4) Ringkasan + cek
tgl = pd.to_datetime(df["tanggal"])                                # hanya untuk ringkasan; kolom asli tetap
per_bulan = {k: int(v) for k, v in tgl.dt.strftime("%Y-%m").value_counts().sort_index().items()}
versi = sorted(df["versi"].astype(str).unique())
judul("1.1", "Muat data mentah", masukan=f"{rel(PATH['xlsx'])} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"Kolom dibuang  : {', '.join(map(str, buang)) or '-'}")
print(f"Rentang tanggal: {tgl_id(tgl.min())} – {tgl_id(tgl.max())}")
print(f"Versi aplikasi : {', '.join(versi)}\n\n{'Bulan':<16}{'Ulasan':>7}")
for k, v in per_bulan.items():
    print(f"{BULAN[int(k[5:]) - 1] + ' ' + k[:4]:<16}{fmt_id(v):>7}")
print(f"{'Total':<16}{fmt_id(sum(per_bulan.values())):>7}")
cek("jumlah baris", len(df), T["baris"])
cek("nama dan urutan kolom", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(map(str, df.columns)), "sesuai")
cek("rentang tanggal", (str(tgl.min().date()), str(tgl.max().date())), T["tanggal"])
cek("nilai unik versi", versi, T["versi"])
cek("ulasan per bulan", per_bulan, T["per_bulan"])
cek("parquet identik dengan isi .xlsx", "ya" if identik else "tidak", "ya")
selesai()

## Tahap 1.2 — Tulis `lib/wondr_teks.py` dan audit kualitas data *(audit: program saja; data tidak diubah)*

**Dasar naskah:** Bab III tahap 1 (data mentah tidak diubah) dan tahap 2 (alasan membuang `nama`, `balasan_dev`, `likes`); pola teks sama dengan Tahap 4 (Bab II 2.2.2, 2.2.4, 2.2.5). Audit tidak dibahas di naskah.

**Package:** `json`, `importlib` (bawaan Python); `pandas` (olah tabel); `scipy.stats.spearmanr` (korelasi *likes*–*rating*); `lib.wondr_teks` (pola `EMO`, `SIMBOL`, `HURUF`, `BACA`; memakai `re`, `unicodedata`, `numpy`, `pandas`); `lib.bantu`.

Sel 1/2 menulis kode acuan Bagian 7 `CLAUDE.md` **persis** ke `lib/wondr_teks.py` (fungsi saja). Sel 2/2 mengaudit data tanpa mengubahnya.

In [ ]:
%%writefile lib/wondr_teks.py
# lib/wondr_teks.py -- kode acuan Bagian 7 CLAUDE.md, disalin persis (Tahap 1.2).
# Dipakai Tahap 1.2 (audit), 4 (pre-processing), 5 (deduplikasi), dan 6 (pembagian data).
import re, unicodedata
import numpy as np, pandas as pd

# -- Pola teks (Tahap 1.2 dan Tahap 4 memakai pola yang sama) --
EMO = re.compile("[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2600-\u27BF\u2B00-\u2BFF"
                 "\u2300-\u23FF\u2190-\u21FF\u2900-\u297F\u3030\u303D\u3297\u3299"
                 "\u00A9\u00AE\u203C\u2049\u2122\u2139\u24C2\u25AA-\u25FE\uFE0F\u200D"
                 "\U000E0020-\U000E007F]")
SIMBOL = re.compile("[\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D\U0001F3FB-\U0001F3FF]")
HURUF = re.compile(r"([^\W\d_])\1{2,}")      # huruf yang berulang >= 3 kali (hanya huruf)
BACA = re.compile(r"[!?.,;:]{2,}")           # tanda baca berurutan >= 2

LANGKAH = [  # (nama, fungsi) -- urutan Tahap 4, jangan diubah
    ("1. Normalisasi Unicode NFKC", lambda s: unicodedata.normalize("NFKC", s)),
    ("2. Hapus emoji dan simbol", lambda s: SIMBOL.sub(" ", EMO.sub(" ", s))),
    ("3. Case folding", lambda s: s.lower()),
    ("4. Huruf berulang >=3 -> 2", lambda s: HURUF.sub(r"\1\1", s)),
    ("5. Tanda baca berulang -> 1", lambda s: BACA.sub(lambda m: m.group(0)[0], s)),
    ("6. Rapikan spasi", lambda s: re.sub(r"\s+", " ", s).strip()),
]

def bersihkan(s):
    s = str(s)
    for _, f in LANGKAH:
        s = f(s)
    return s

# -- Tahap 5: deduplikasi --
def kunci_dedup(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s))   # sisakan a-z, 0-9, spasi
    s = re.sub(r"(.)\1+", r"\1", s)          # karakter berurutan yang sama -> satu
    return re.sub(r"\s+", " ", s).strip()

def deduplikasi(d):
    d = d.copy()
    d["dedup_key"] = d["text_clean"].map(kunci_dedup)
    d = d[d["dedup_key"] != ""]
    suara = (d.groupby(["dedup_key", "label"]).size().unstack(fill_value=0)
               .reindex(columns=[0, 1], fill_value=0))
    suara.columns = ["n_np", "n_p"]
    seri = suara["n_np"] == suara["n_p"]
    suara["mayor"] = (suara["n_p"] > suara["n_np"]).astype(int)
    sah = suara[~seri]
    x = d[d["dedup_key"].isin(sah.index)].copy()
    x = x[x["label"] == x["dedup_key"].map(sah["mayor"])]          # anggota berlabel mayoritas
    frek = x.groupby(["dedup_key", "text_clean"]).size().rename("_f").reset_index()
    x = x.merge(frek, on=["dedup_key", "text_clean"])
    wakil = (x.sort_values(["dedup_key", "_f", "review_id"], ascending=[True, False, True])
               .groupby("dedup_key").head(1)["review_id"])           # teks tersering, lalu id terkecil
    hasil = d[d["review_id"].isin(set(wakil))].copy()
    hasil["label"] = hasil["dedup_key"].map(sah["mayor"]).astype(int)
    return hasil.sort_values("review_id").reset_index(drop=True), suara, seri

# -- Tahap 6: pembagian data berbasis grup --
KATA_FUNGSI = {"dan", "yang", "nya", "ini", "itu", "ya", "sih", "deh", "juga", "aja", "saja", "nih",
               "kok", "dong", "lah", "di", "ke", "dari", "untuk", "pada", "dengan", "yg", "jg", "utk", "dgn"}

def kunci_grup(s):                      # hanya untuk pengelompokan; teks tidak diubah
    k = set(str(s).split())
    inti = k - KATA_FUNGSI
    return " ".join(sorted(inti)) if inti else " ".join(sorted(k))

def kelompok_panjang(n_kata):
    return "1-2" if n_kata <= 2 else ("3-10" if n_kata <= 10 else ">10")

PROPORSI = {"latih": 0.70, "validasi": 0.15, "uji": 0.15}   # urutan kunci menentukan pemecah seri

def bagi_data(dd, seed=42):
    dd = dd.copy()
    dd["group_key"] = dd["text_clean"].map(kunci_grup)
    dd["n_kata"] = dd["text_clean"].str.split().str.len()
    label_id_kecil = dd.sort_values("review_id").groupby("group_key")["label"].first()
    g = (dd.groupby("group_key")
           .agg(n=("review_id", "size"), n_p=("label", "sum"), kata=("n_kata", "median"))
           .reset_index())
    mayor = np.where(g.n_p * 2 > g.n, 1, np.where(g.n_p * 2 < g.n, 0, -1))
    g["label_grup"] = np.where(mayor == -1, g.group_key.map(label_id_kecil).to_numpy(), mayor).astype(int)
    g["kelompok_panjang"] = g["kata"].map(kelompok_panjang)
    rng = np.random.default_rng(seed)                     # satu generator untuk semua strata
    alokasi = {}
    for _, sub in g.groupby(["label_grup", "kelompok_panjang"], sort=True):
        sub = sub.iloc[rng.permutation(len(sub))]
        total = int(sub.n.sum())
        target = {b: p * total for b, p in PROPORSI.items()}
        terisi = {b: 0 for b in PROPORSI}
        for kunci, n in zip(sub.group_key, sub.n):
            b = min(terisi, key=lambda b: terisi[b] / target[b])   # keterisian terkecil
            alokasi[kunci] = b
            terisi[b] += int(n)
    dd["split"] = dd["group_key"].map(alokasi)
    dd["kelompok_panjang"] = dd["n_kata"].map(kelompok_panjang)
    return dd, g

In [ ]:
# ══ Tahap 1.2 — Audit kualitas data (sel 2/2; program saja, data tidak diubah) ══
# Package: json, importlib (bawaan Python); pandas; scipy.stats.spearmanr; lib.wondr_teks (pola teks)
import json, importlib
import pandas as pd
from scipy.stats import spearmanr
import lib.wondr_teks as wt
from lib.bantu import _tampil                              # format angka Indonesia untuk tabel
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["1.2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
df = pd.read_parquet(masuk)
u, r = df["ulasan"].astype(str), df["rating"]
n_kata, teks = u.str.split().str.len(), u.str.lower().str.strip()    # kata = dipisah spasi

# 2) Pemeriksaan (data hanya dibaca); label sementara = rating >= tau, hanya untuk audit
dibalas, huruf = df["balasan_dev"].notna(), u.str.lower().map(lambda s: bool(wt.HURUF.search(s)))
n_lbl = (r >= CFG["tau"]).astype(int).groupby(teks).nunique()
med = [float(n_kata[r == k].median()) for k in range(1, 6)]
emoji = int(u.map(lambda s: bool(wt.EMO.search(s) or wt.SIMBOL.search(s))).sum())
AUDIT = [  # (temuan, nilai aktual, kunci TARGET, toleransi, tahap pemakai)
    ("Nilai kosong per kolom", {k: int(v) for k, v in df.isna().sum().items() if v}, "kosong", 0, "2"),
    ("Dibalas pengembang (%), rating 1–4 · 5",
     {"1-4": dibalas[r <= 4].mean() * 100, "5": dibalas[r == 5].mean() * 100}, "dibalas_pct", 0.05, "2"),
    ("Likes bernilai nol (%)", (df["likes"] == 0).mean() * 100, "likes_nol_pct", 0.05, "2"),
    ("Spearman likes–rating", spearmanr(df["likes"], r).statistic, "spearman", 5e-5, "2"),
    ("Nama unik", int(df["nama"].nunique()), "nama_unik", 0, "2"),
    ("Rating 3 >= 8 kata (jumlah · total)", (int(((r == 3) & (n_kata >= 8)).sum()), int((r == 3).sum())),
     "r3_panjang", 0, "3"),
    ("Median kata per rating 1–5", [int(m) if m == int(m) else m for m in med], "median_kata", 0, "6.1"),
    ("Baris berisi emoji atau simbol", emoji, "emoji_simbol", 0, "4"),
    ("Huruf berulang >= 3 (baris · %)", (int(huruf.sum()), huruf.mean() * 100), "huruf_ulang", 0.05, "4"),
    ("Tanda baca berulang (baris)", int(u.map(lambda s: bool(wt.BACA.search(s))).sum()), "baca_ulang", 0, "4"),
    ("Duplikat persis (% baris)", df["ulasan"].duplicated().mean() * 100, "duplikat_pct", 0.05, "5"),
    ("Teks berlabel ganda (teks · baris)", (int((n_lbl == 2).sum()), int(teks.isin(n_lbl[n_lbl == 2].index).sum())),
     "label_ganda", 0, "5"),
]

# 3) Simpan tabel (angka mentah, JSON), tampilkan temuan -> angka -> tahap pemakai, lalu cek
keluar = simpan_tabel(pd.DataFrame([{"temuan": t, "nilai": json.dumps(v), "target": json.dumps(T[k]), "tahap": th}
                                    for t, v, k, _, th in AUDIT]), "tahap01_audit")
judul("1.2", "Audit kualitas data", masukan=[f"{rel(masuk)} ({fmt_id(len(df))} baris)", "lib/wondr_teks.py"],
      keluaran=f"{rel(keluar)} ({len(AUDIT)} baris)")
print(f"{'Temuan':<40}{'Angka':<28}Tahap")
for t, v, _, tol, th in AUDIT:
    print(f"{t:<40}{_tampil(v, 1 if tol == 0.05 else 4):<28}{th}")
cek("jumlah baris tetap", len(df), TARGET["1.1"]["baris"])
for t, v, k, tol, _ in AUDIT:
    cek(t, v, T[k], toleransi=tol)
selesai()

## Tahap 2 — Seleksi atribut

**Dasar naskah:** Bab III tahap 2 (nomor urut sebagai identitas; atribut yang digunakan hanya teks ulasan dan *rating*; nama, balasan pengembang, dan *likes* dihapus; tanggal dan versi hanya informasi pendukung); Bab II subbab 2.4.2. `review_id` dibuat di sini (`CLAUDE.md` Bagian 10, keputusan no. 7).

**Package:** `pandas` (olah tabel), `pyarrow` (mesin baca/simpan .parquet); `lib.bantu`.

In [ ]:
# ══ Tahap 2 — Seleksi atribut ══
# Package: pandas (olah tabel), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
mulai()
T = TARGET["2"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap01_mentah.parquet"
mentah = pd.read_parquet(masuk)

# 2) Nomor urut sebagai identitas, lalu pilih atribut (ringkasan kalimat Bab III tahap 2)
ATRIBUT = [  # (atribut, peran, alasan)
    ("review_id", "dibuat", "nomor urut sesuai urutan baris data mentah, sebagai identitas"),
    ("ulasan", "dipakai", "teks ulasan: satu-satunya masukan model"),
    ("rating", "dipakai", "sumber label awal (Tahap 3), bukan fitur"),
    ("tanggal", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("versi", "disimpan", "informasi pendukung (subbab 2.4.2)"),
    ("nama", "dibuang", "nama pengguna; dihapus untuk menjaga privasi"),
    ("balasan_dev", "dibuang", "ditulis setelah ulasan; isinya dapat membocorkan label"),
    ("likes", "dibuang", "tidak tersedia pada saat prediksi"),
]
pakai = [a for a, peran, _ in ATRIBUT if peran != "dibuang"]
df = mentah.assign(review_id=range(1, len(mentah) + 1))[pakai]

# 3) Simpan
keluar = PATH["hasil"] / "tahap02_atribut.parquet"
df.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("2", "Seleksi atribut", masukan=f"{rel(masuk)} ({fmt_id(len(mentah))} baris, {mentah.shape[1]} kolom)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(df))} baris, {df.shape[1]} kolom)")
print(f"{'Atribut':<13}{'Peran':<10}Alasan")
for a, peran, alasan in ATRIBUT:
    print(f"{a:<13}{peran:<10}{alasan}")
tercatat = {a for a, _, _ in ATRIBUT} - {"review_id"} == set(mentah.columns)
ya = lambda b: "ya" if b else "tidak"
cek("jumlah baris", len(df), T["baris"])
cek("kolom dan urutannya", "sesuai" if list(df.columns) == T["kolom"] else ", ".join(df.columns), "sesuai")
cek("semua kolom data mentah tercatat di tabel", ya(tercatat), "ya")
cek("review_id = 1, 2, ..., n (unik, urut)", ya(df["review_id"].tolist() == list(range(1, len(df) + 1))), "ya")
cek("isi ulasan, rating, tanggal, versi tidak berubah", ya(df[pakai[1:]].equals(mentah[pakai[1:]])), "ya")
selesai()

## Tahap 3 — Pelabelan awal (*top-2-box*)

**Dasar naskah:** Bab III tahap 3; pers. (2.1)–(2.2) dengan τ = 4; Tabel 2.8; Morgan dan Rego (2006). Kolom bernama `label` agar kode acuan `lib/wondr_teks.py` tidak diubah; namanya menjadi `label_awal` di Tahap 9.3.

**Package:** `pandas` (olah tabel), `matplotlib` (grafik), `pyarrow` (mesin baca/simpan .parquet); `lib.bantu`. Warna kelas: slot 1–2 palet kategori yang lolos validasi buta warna (Positif biru `#2a78d6`, Non-Positif oranye `#eb6834`), dipakai tetap di semua gambar.

In [ ]:
# ══ Tahap 3 — Pelabelan awal (top-2-box) ══
# Package: pandas (olah tabel), matplotlib (grafik), pyarrow (mesin baca/simpan .parquet)
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
mulai()
T = TARGET["3"]
WARNA = {1: "#2a78d6", 0: "#eb6834"}          # slot kategori 1-2 palet tervalidasi (CVD aman); tetap per kelas

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap02_atribut.parquet"
df = pd.read_parquet(masuk)

# 2) Label awal pers. (2.1)-(2.2): rating >= tau -> 1 (Positif), selain itu 0 (Non-Positif); tidak ada baris dihapus
df["label"] = (df["rating"] >= CFG["tau"]).astype(int)
keluar = PATH["hasil"] / "tahap03_label.parquet"
df.to_parquet(keluar, index=False)

# 3) Gambar: jumlah ulasan per rating, diwarnai per kelas
per = df.groupby("rating").agg(n=("review_id", "size"), kelas=("label", "first"))
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(per.index, per.n, width=0.6, color=[WARNA[k] for k in per.kelas], zorder=2)
for r, n in per.n.items():
    ax.text(r, n + 60, fmt_id(int(n)), ha="center", va="bottom", fontsize=9, color="#52514e")
ax.set(xlabel="Rating", ylabel="Jumlah ulasan", xticks=per.index, ylim=(0, per.n.max() * 1.12))
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
ax.legend(handles=[Patch(color=WARNA[k], label=CFG["kelas"][k]) for k in (1, 0)], frameon=False, loc="upper left")
gambar = simpan_gambar(fig, "tahap03_distribusi_rating")

# 4) Ringkasan + cek
kelas = {CFG["kelas"][k]: (int((df.label == k).sum()), (df.label == k).mean() * 100) for k in (1, 0)}
judul("3", "Pelabelan awal (top-2-box)", masukan=f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(df))} baris, kolom baru: label)", rel(gambar)])
print(f"{'Rating':<8}{'Jumlah':>8}  Kelas")
for r, (n, k) in per.iterrows():
    print(f"{r:<8}{fmt_id(int(n)):>8}  {CFG['kelas'][k]}")
for nama, (n, pct) in kelas.items():
    print(f"Total {nama:<12}{fmt_id(n):>7}  ({fmt_id(pct, 2)}%)")
cek("jumlah baris tetap (tidak ada yang dihapus)", len(df), TARGET["2"]["baris"])
cek("jumlah per rating", {int(r): int(n) for r, n in per.n.items()}, T["per_rating"])
cek("Positif (jumlah · %)", kelas["Positif"], T["positif"], toleransi=0.005)
cek("Non-Positif (jumlah · %)", kelas["Non-Positif"], T["non_positif"], toleransi=0.005)
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

## Tahap 4 — *Pre-processing*

**Dasar naskah:** Bab III tahap 4; Bab II subbab 2.2 (Tabel 2.1–2.7). Enam langkah `LANGKAH` dari `lib/wondr_teks.py` dijalankan berurutan, lalu teks kosong dibuang. Tanpa *stemming*, tanpa *stopword removal*, dan tanpa normalisasi kamus.

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (memakai `re`, `unicodedata`); `lib.bantu`. Masukan uji contoh Bab II ditulis sebagai *escape* Unicode.

In [ ]:
# ══ Tahap 4 — Pre-processing ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (LANGKAH, bersihkan; memakai re, unicodedata)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["4"]

# 1) Uji bersihkan() dengan contoh Bab II (Tabel 2.1-2.7); masukan ditulis sebagai escape Unicode
CONTOH = ["\U0001D401\U0001D400\U0001D406\U0001D414\U0001D412\U0001D412\U0001D412 bgt!!! \u2605\u2605 tp "
          "kadang\u00b2 error, nunggu lamaaa", "\uff4d\uff41\uff4e\uff54\uff41\uff50"]
uji = [wt.bersihkan(x) for x in CONTOH]

# 2) Baca masukan, jalankan 6 langkah berurutan sambil menghitung ulasan yang berubah
masuk = PATH["hasil"] / "tahap03_label.parquet"
df = pd.read_parquet(masuk)
s, berubah = df["ulasan"].astype(str), []
for i, (_, f) in enumerate(wt.LANGKAH):
    t = s.map(f)
    berubah.append(int((t != s).sum()))
    if i == 3:                                              # langkah 4: simpan contoh sebelum -> sesudah
        contoh4 = pd.DataFrame({"id": df["review_id"], "a": s, "b": t})[t != s].head(3)
    s = t
df["text_clean"] = s

# 3) Buang teks kosong (jumlah sebelum dan sesudah ditampilkan), lalu simpan
kosong, bersih = df[df["text_clean"] == ""], df[df["text_clean"] != ""].reset_index(drop=True)
keluar = PATH["hasil"] / "tahap04_bersih.parquet"
bersih.to_parquet(keluar, index=False)

# 4) Ringkasan + cek
judul("4", "Pre-processing", masukan=f"{rel(masuk)} ({fmt_id(len(df))} baris)",
      keluaran=f"{rel(keluar)} ({fmt_id(len(bersih))} baris, kolom baru: text_clean)")
print(f"{'Langkah':<32}{'Ulasan berubah':>15}")
for (nama, _), n in zip(wt.LANGKAH, berubah):
    print(f"{nama:<32}{fmt_id(n):>15}")
r_kosong = {int(k): int(v) for k, v in kosong["rating"].value_counts().items()}
print(f"\nTeks kosong dibuang: {fmt_id(len(kosong))} ({', '.join(f'rating {k}: {v}' for k, v in r_kosong.items())})"
      f"  →  {fmt_id(len(df))} menjadi {fmt_id(len(bersih))} ulasan")
print("Contoh langkah 4 saja (huruf berulang >= 3 -> 2); tanda baca berulang baru dipadatkan di langkah 5:")
for i, a, b in contoh4.itertuples(index=False):                # teks utuh, tanpa pemotongan
    kata = "; ".join(f'"{x}" → "{y}"' for x, y in zip(a.split(), b.split()) if x != y)
    print(f"  [{i}] kata yang berubah : {kata}")
    print(f"       hasil 6 langkah  : {bersih.loc[bersih['review_id'] == i, 'text_clean'].iloc[0]}")
cek("uji contoh Bab II (Tabel 2.1-2.7)", uji, T["contoh_bab2"])
cek("ulasan berubah per langkah", berubah, T["berubah"])
cek("teks kosong dibuang", len(kosong), T["kosong"])
cek("rating teks kosong", r_kosong, T["kosong_rating"])
cek("jumlah setelah pre-processing", len(bersih), T["hasil"])
tc = bersih["text_clean"]                                      # invarian: aturan berlaku di seluruh teks
cek("sisa huruf berulang · tanda baca berulang · emoji", [int(tc.map(lambda x: bool(p.search(x))).sum())
    for p in (wt.HURUF, wt.BACA, wt.EMO)], [0, 0, 0])
sama = (bersih["ulasan"].map(wt.bersihkan) == tc).all()
cek("text_clean = bersihkan(ulasan) di semua baris", "ya" if sama else "tidak", "ya")
selesai()

## Tahap 5 — Deduplikasi

**Dasar naskah:** Bab III tahap 5; Bab II subbab 2.4.1 dan Tabel 2.11 (label mayoritas; kunci seri dikeluarkan; wakil dipilih berdasarkan kesesuaian dengan label mayoritas, teks tersering, lalu nomor urut terkecil). Tabel kunci seri ditampilkan sebagai *program saja* (`CLAUDE.md` Bagian 10, keputusan no. 10).

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (`kunci_dedup`, `deduplikasi`); `lib.bantu`.

In [ ]:
# ══ Tahap 5 — Deduplikasi ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (kunci_dedup, deduplikasi)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["5"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap04_bersih.parquet"
d = pd.read_parquet(masuk)

# 2) Deduplikasi (kode acuan): label mayoritas, kunci seri dikeluarkan, wakil = teks tersering lalu id terkecil
hasil, suara, seri = wt.deduplikasi(d)
keluar = PATH["hasil"] / "tahap05_dedup.parquet"
hasil.to_parquet(keluar, index=False)

# 3) Rincian alur, dihitung dari tabel suara per kunci; simpan CSV (angka mentah)
kunci, sah = d["text_clean"].map(wt.kunci_dedup), suara[~seri]
ALUR = {"ulasan masuk": len(d), "kunci kosong (dibuang)": int((kunci == "").sum()),
        "kunci deduplikasi": len(suara), "kunci seri": int(seri.sum()),
        "dibuang: salinan berlabel sama": int((sah[["n_np", "n_p"]].max(axis=1) - 1).sum()),
        "dibuang: anggota kalah suara": int(sah[["n_np", "n_p"]].min(axis=1).sum()),
        "dibuang: anggota kunci seri": int(suara.loc[seri, ["n_np", "n_p"]].to_numpy().sum()),
        "dokumen hasil": len(hasil)}
alur = simpan_tabel(pd.DataFrame({"langkah": list(ALUR), "jumlah": list(ALUR.values())}), "tahap05_alur")

# 4) Ringkasan + cek
dibuang = [len(d) - ALUR["kunci kosong (dibuang)"] - len(hasil)] + [v for k, v in ALUR.items() if k.startswith("dibuang")]
pos = (d["label"].mean() * 100, hasil["label"].mean() * 100)
judul("5", "Deduplikasi", masukan=f"{rel(masuk)} ({fmt_id(len(d))} baris)",
      keluaran=[f"{rel(keluar)} ({fmt_id(len(hasil))} baris, kolom baru: dedup_key)", rel(alur)])
for k, v in ALUR.items():
    print(f"{k:<34}{fmt_id(v):>7}")
print(f"Proporsi Positif                  {fmt_id(pos[0], 1)}% → {fmt_id(pos[1], 1)}%")
print("\nKunci seri yang dikeluarkan (program saja; rating anggotanya saling bertentangan):")
anggota = d.assign(k=kunci)[kunci.isin(suara[seri].index)].sort_values("review_id")
for k, g in anggota.groupby("k"):
    print(f"  {k:<20}" + " · ".join(f"[{i}] rating {r}" for i, r in zip(g["review_id"], g["rating"])))
cek("kunci kosong dibuang", ALUR["kunci kosong (dibuang)"], 0)
cek("kunci · kunci seri · ulasan seri", (len(suara), int(seri.sum()), dibuang[3]),
    (T["kunci"], T["kunci_seri"], T["ulasan_seri"]))
cek("jumlah dokumen", len(hasil), T["dokumen"])
cek("dibuang (total · salinan · kalah suara · seri)", dibuang, T["dibuang"])
cek("total dibuang = jumlah ketiga rincian", "ya" if dibuang[0] == sum(dibuang[1:]) else "tidak", "ya")
cek("Positif (%) sebelum · sesudah", pos, T["positif_pct"], toleransi=0.05)
cek("satu dokumen per kunci", "ya" if hasil["dedup_key"].is_unique else "tidak", "ya")
selesai()

## Tahap 6.1 — Pembagian data

**Dasar naskah:** Bab III tahap 6; pers. (2.6); Tabel 2.12–2.13. Latih 70%, validasi 15%, uji 15%; grup berkunci sama ditempatkan pada himpunan yang sama; stratifikasi pada tingkat grup berdasarkan label mayoritas grup dan kelompok panjang (1–2, 3–10, >10 kata); grup setiap strata diacak dengan *seed* 42 lalu dialokasikan ke himpunan dengan tingkat pengisian terendah.

**Package:** `importlib` (bawaan Python); `pandas`; `lib.wondr_teks` (`bagi_data`, `kunci_grup`, `kelompok_panjang`; memakai `numpy`); `lib.bantu`.

In [ ]:
# ══ Tahap 6.1 — Pembagian data ══
# Package: importlib (bawaan Python); pandas; lib.wondr_teks (bagi_data, kunci_grup, kelompok_panjang)
import importlib
import pandas as pd
import lib.wondr_teks as wt
importlib.reload(wt)                                        # pakai isi lib/wondr_teks.py yang terbaru
mulai()
T = TARGET["6.1"]
KOLOM = ["review_id", "ulasan", "text_clean", "rating", "label", "group_key", "dedup_key", "kelompok_panjang"]

# 1) Baca masukan
masuk = PATH["hasil"] / "tahap05_dedup.parquet"
dd = pd.read_parquet(masuk)

# 2) Bagi berbasis grup pers. (2.6), stratifikasi label grup x kelompok panjang (kode acuan; seed dari CFG)
bagi, grup = wt.bagi_data(dd, seed=CFG["seed_bagi"])

# 3) Simpan tiga bagian
keluar = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in CFG["proporsi"]}
for b, p in keluar.items():
    bagi.loc[bagi["split"] == b, KOLOM].reset_index(drop=True).to_parquet(p, index=False)

# 4) Ringkasan + cek
hit = {b: (int((bagi.split == b).sum()), int(((bagi.split == b) & (bagi.label == 0)).sum()),
           int(((bagi.split == b) & (bagi.label == 1)).sum())) for b in CFG["proporsi"]}
judul("6.1", "Pembagian data", masukan=f"{rel(masuk)} ({fmt_id(len(dd))} dokumen)",
      keluaran=[f"{rel(p)} ({fmt_id(hit[b][0])} dokumen)" for b, p in keluar.items()])
print(f"Grup: {fmt_id(len(grup))} dari {fmt_id(len(dd))} dokumen "
      f"({fmt_id(int((grup.n > 1).sum()))} grup berisi lebih dari satu dokumen)\n")
print(f"{'Bagian':<10}{'Dokumen':>8}{'Porsi':>8}{'Target':>8}{'Non-Positif':>13}{'Positif':>9}")
for b, p in CFG["proporsi"].items():
    n, n0, n1 = hit[b]
    print(f"{b:<10}{fmt_id(n):>8}{fmt_id(n / len(bagi) * 100, 1) + '%':>8}{fmt_id(p * 100, 0) + '%':>8}"
          f"{fmt_id(n0):>13}{fmt_id(n1):>9}")
cek("PROPORSI di lib/wondr_teks.py = CFG", wt.PROPORSI, CFG["proporsi"])
cek("jumlah grup", len(grup), T["grup"])
for b in CFG["proporsi"]:
    cek(f"{b} (dokumen · Non-Positif · Positif)", hit[b], T[b])
cek("semua dokumen mendapat satu bagian", "ya" if bagi["split"].notna().all() and len(bagi) == len(dd) else "tidak", "ya")
selesai()

## Tahap 6.2 — Cek kebocoran dan keseimbangan

**Dasar naskah:** Bab III tahap 6 (setelah pembagian dipastikan tidak ada kunci pengelompokan, teks, dan kunci deduplikasi yang muncul pada lebih dari satu himpunan); Bab II subbab 2.4.2–2.4.4. Kedua sel hanya **membaca** berkas Tahap 6.1; *seed*, proporsi, dan algoritma pembagian tidak diubah.

**Package:** `pandas` (olah tabel); `scipy.stats.chi2_contingency` (uji khi-kuadrat); `lib.bantu`.

Sel 1/2: kebocoran `group_key`, `dedup_key`, `text_clean`, kelengkapan dokumen, dan posisi kembaran-dekat. Sel 2/2: proporsi label dan kelompok panjang per bagian, uji khi-kuadrat, dan tabel strata. **Mulai di sini data uji dikunci.**

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 1/2: kebocoran dan kelengkapan) ══
# Package: pandas (olah tabel, baca .parquet)
import pandas as pd
mulai()
T, BAGIAN = TARGET["6.2"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dari berkas Tahap 6.1 (pembagian tidak diubah) dan hasil deduplikasi
masuk = {b: PATH["hasil"] / f"tahap06_{b}.parquet" for b in BAGIAN}
df = pd.concat([pd.read_parquet(p).assign(bagian=b) for b, p in masuk.items()], ignore_index=True)
dedup = pd.read_parquet(PATH["hasil"] / "tahap05_dedup.parquet")

# 2) Kebocoran: kunci yang muncul di lebih dari satu bagian
KUNCI = ("group_key", "dedup_key", "text_clean")
lintas = {k: df.groupby(k)["bagian"].nunique() for k in KUNCI}
bocor = {k: s[s > 1] for k, s in lintas.items()}

# 3) Ringkasan + cek
judul("6.2", "Cek kebocoran (sel 1/2)", masukan=[f"{rel(p)} ({fmt_id(int((df.bagian == b).sum()))} dokumen)"
                                                for b, p in masuk.items()])
print(f"{'Kunci':<12}{'Nilai unik':>11}{'Di > 1 bagian':>15}")
for k in KUNCI:
    print(f"{k:<12}{fmt_id(len(lintas[k])):>11}{fmt_id(len(bocor[k])):>15}")
for k in KUNCI:                                             # contoh hanya dicetak bila ada kebocoran
    for kunci in bocor[k].index[:5]:
        print(f"  BOCOR {k} {kunci!r}: {sorted(df.loc[df[k] == kunci, 'bagian'].unique())}")
grup_per_bagian = int(df.groupby("bagian")["group_key"].nunique().sum())
for k in KUNCI:
    cek(f"{k} di lebih dari satu bagian", len(bocor[k]), T["bocor"])
cek("jumlah grup per bagian dijumlahkan", grup_per_bagian, TARGET["6.1"]["grup"])
cek("total dokumen · review_id unik", (len(df), int(df["review_id"].nunique())),
    (TARGET["5"]["dokumen"], TARGET["5"]["dokumen"]))
sama = set(df["review_id"]) == set(dedup["review_id"])
cek("review_id = review_id hasil deduplikasi", "ya" if sama else "tidak", "ya")
kd = df.loc[df["review_id"].isin(CFG["kembaran_dekat"]), ["review_id", "bagian"]]
cek("kembaran-dekat berada di uji", int((kd["bagian"] == "uji").sum()), T["kembaran_di_uji"])
selesai()

In [ ]:
# ══ Tahap 6.2 — Cek kebocoran dan keseimbangan (sel 2/2: keseimbangan dan strata) ══
# Package: pandas (olah tabel); scipy.stats.chi2_contingency (uji khi-kuadrat)
import pandas as pd
from scipy.stats import chi2_contingency
mulai()
T, BAGIAN, PANJANG = TARGET["6.2"], list(CFG["proporsi"]), list(CFG["kelompok_panjang"])

# 1) Baca ketiga bagian (pembagian tidak diubah)
df = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet").assign(bagian=b) for b in BAGIAN],
               ignore_index=True)

# 2) Tabel silang bagian x label dan bagian x kelompok panjang, lalu uji khi-kuadrat
x_label = pd.crosstab(df["bagian"], df["label"]).loc[BAGIAN]
x_panjang = pd.crosstab(df["bagian"], df["kelompok_panjang"]).loc[BAGIAN, PANJANG]
p_label, p_panjang = chi2_contingency(x_label)[1], chi2_contingency(x_panjang)[1]
pct_np = x_label[0] / x_label.sum(axis=1) * 100
pct_pj = x_panjang.div(x_panjang.sum(axis=1), axis=0) * 100

# 3) Simpan tabel strata (label x kelompok panjang) per bagian, angka mentah
strata = pd.crosstab([df["label"].map(CFG["kelas"]), df["kelompok_panjang"]], df["bagian"])[BAGIAN]
keluar = simpan_tabel(strata.reset_index().rename(columns={"label": "kelas"}), "tahap06_strata")

# 4) Ringkasan + cek
judul("6.2", "Cek keseimbangan (sel 2/2)", masukan="hasil/tahap06_{latih,validasi,uji}.parquet", keluaran=rel(keluar))
print(f"{'Bagian':<10}{'Non-Positif':>12}" + "".join(f"{'kata ' + k:>12}" for k in PANJANG))
for b in BAGIAN:
    sel = [fmt_id(pct_np[b], 1)] + [fmt_id(pct_pj.loc[b, k], 1) for k in PANJANG]
    print(f"{b:<10}" + "".join(f"{x + '%':>12}" for x in sel))
print(f"\nKhi-kuadrat: bagian x label p = {fmt_id(p_label)} · bagian x kelompok panjang p = {fmt_id(p_panjang)}")
print(f"\nDokumen per strata\n{strata.to_string()}")
cek("Non-Positif (%) latih · validasi · uji", [pct_np[b] for b in BAGIAN], T["non_positif_pct"], toleransi=0.05)
for b in BAGIAN:
    cek(f"kelompok panjang (%) {b}", list(pct_pj.loc[b]), T["panjang_pct"][b], toleransi=0.05)
cek("p khi-kuadrat label · panjang", (p_label, p_panjang), (T["p_label"], T["p_panjang"]), toleransi=5e-5)
print("\nMulai di sini data uji dikunci: label dan metriknya tidak dipakai sampai Tahap 14 "
      "(kecuali teksnya diekspor ke lembar anotasi di Tahap 9.1).")
selesai()

## Tahap 7 — Tokenisasi

**Dasar naskah:** Bab III tahap 7; Bab II subbab 2.5 dan 2.7; pers. (2.7)–(2.9); Tabel 2.14. *Tokenizer* bawaan `indobenchmark/indobert-base-p1`; *fertility*, r_UNK, dan L_max dihitung **hanya dari data latih** (q = 100, b = 32); panjang dokumen = token subkata + [CLS] + [SEP]; dokumen validasi/uji yang lebih panjang dari L_max hanya dideskripsikan (akan terpotong).

**Package:** `math` (bawaan Python); `numpy`; `pandas`; `transformers.AutoTokenizer`; `matplotlib`; `lib.bantu`.

Sel 1/2: token, statistik, dan L_max → `hasil/tahap07_tokenisasi.json`. Sel 2/2: histogram panjang token data latih → `bab4/gambar/tahap07_panjang_token.png`.

In [ ]:
# ══ Tahap 7 — Tokenisasi (sel 1/2: token subkata, statistik data latih, L_max) ══
# Package: math (bawaan Python); numpy; pandas; transformers.AutoTokenizer (tokenizer IndoBERT)
import math
import numpy as np, pandas as pd
from transformers import AutoTokenizer
mulai()
T, BAGIAN = TARGET["7"], list(CFG["proporsi"])

# 1) Baca ketiga bagian dan tokenizer bawaan checkpoint (harus sama dengan tahap prapelatihan)
data = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet") for b in BAGIAN}
tok = AutoTokenizer.from_pretrained(CFG["checkpoint"])

# 2) Token subkata per dokumen (tanpa [CLS]/[SEP]); panjang dokumen = subkata + 2
sub = {b: [tok.tokenize(t) for t in d["text_clean"]] for b, d in data.items()}
panjang = {b: np.array([len(s) + 2 for s in sub[b]]) for b in BAGIAN}
cocok = all((panjang[b] == [len(x) for x in tok(list(data[b]["text_clean"]))["input_ids"]]).all() for b in BAGIAN)

# 3) Statistik HANYA dari data latih: fertility (2.7), r_UNK (2.8), L_max (2.9); validasi dan uji hanya deskripsi
n_sub, n_kata = sum(map(len, sub["latih"])), int(data["latih"]["text_clean"].str.split().str.len().sum())
n_unk = sum(s.count(tok.unk_token) for s in sub["latih"])
F, r_unk, P_q = n_sub / n_kata, n_unk / n_sub, float(np.percentile(panjang["latih"], CFG["q"]))
L_max = CFG["b"] * math.ceil(P_q / CFG["b"])
potong = {b: data[b].loc[panjang[b] > L_max, "review_id"].tolist() for b in BAGIAN}
stat = {b: {"n": len(p), "median": float(np.median(p)), "p95": float(np.percentile(p, 95)), "maks": int(p.max()),
            "terpotong": len(potong[b])} for b, p in panjang.items()}
keluar = simpan_json({"checkpoint": CFG["checkpoint"], "kosakata": tok.vocab_size, "n_subkata": n_sub,
                      "n_kata": n_kata, "n_unk": n_unk, "F": F, "r_UNK": r_unk, "P_q": P_q, "L_max": L_max,
                      "statistik": stat, "terpotong_id": potong, "panjang_latih": panjang["latih"].tolist()},
                     PATH["hasil"] / "tahap07_tokenisasi.json")

# 4) Ringkasan + cek
judul("7", "Tokenisasi (sel 1/2)", masukan=["hasil/tahap06_{latih,validasi,uji}.parquet", f"tokenizer {CFG['checkpoint']}"],
      keluaran=rel(keluar))
print(f"{'Bagian':<10}{'n':>7}{'Median':>8}{'P95':>7}{'Maks':>6}{'Terpotong':>11}")
for b, s in stat.items():
    print(f"{b:<10}{fmt_id(s['n']):>7}{fmt_id(s['median'], 0):>8}{fmt_id(s['p95'], 0):>7}{s['maks']:>6}{s['terpotong']:>11}")
print(f"\nData latih: F = {fmt_id(F)} subkata/kata · r_UNK = {fmt_id(r_unk, 6)} ({fmt_id(n_unk)} dari {fmt_id(n_sub)} token)")
print(f"L_max = {CFG['b']}·⌈P{CFG['q']} / {CFG['b']}⌉ = {CFG['b']}·⌈{fmt_id(P_q, 0)} / {CFG['b']}⌉ = {L_max}")
cek("panjang = subkata + 2 (sesuai tokenizer)", "ya" if cocok else "tidak", "ya")
cek("L_max (naskah: 128)", L_max, T["L_max"])
cek("dokumen terpotong", {b: len(potong[b]) for b in BAGIAN}, T["terpotong"])
cek("review_id yang terpotong", potong["latih"] + potong["validasi"] + potong["uji"], T["terpotong_id"])
selesai()

In [ ]:
# ══ Tahap 7 — Tokenisasi (sel 2/2: histogram panjang token data latih) ══
# Package: numpy; matplotlib (grafik); lib.bantu (baca_json, simpan_gambar)
import numpy as np
import matplotlib.pyplot as plt
mulai()

# 1) Baca hasil sel 1/2
masuk = PATH["hasil"] / "tahap07_tokenisasi.json"
tk = baca_json(masuk)
p, L_max = np.array(tk["panjang_latih"]), tk["L_max"]

# 2) Histogram (satu seri: tanpa legenda) dengan garis L_max
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(p, bins=range(0, L_max + 9, 4), color="#2a78d6", rwidth=0.85, zorder=2)
ax.axvline(L_max, color="#52514e", linewidth=1.2, zorder=3)
ax.text(L_max - 2, ax.get_ylim()[1] * 0.92, f"L_max = {L_max}", ha="right", fontsize=9, color="#52514e")
ax.set(xlabel="Panjang token per dokumen latih (termasuk [CLS] dan [SEP])", ylabel="Jumlah dokumen")
ax.yaxis.set_major_formatter(lambda y, _: fmt_id(int(y)))
ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(colors="#52514e", length=0)
gambar = simpan_gambar(fig, "tahap07_panjang_token")

# 3) Ringkasan + cek
judul("7", "Tokenisasi (sel 2/2)", masukan=rel(masuk), keluaran=rel(gambar))
cek("dokumen latih pada histogram", len(p), TARGET["6.1"]["latih"][0])
cek("semua dokumen latih <= L_max", int((p > L_max).sum()), TARGET["7"]["terpotong"]["latih"])
cek("gambar tersimpan", "ya" if gambar.exists() else "tidak", "ya")
selesai()

### Tahap 7 — Pemeriksaan akhir tokenisasi (tambahan, diminta pengguna)

Hanya membaca hasil Tahap 6.1 dan 7; konfigurasi tokenisasi tidak diubah. Memeriksa: L_max dihitung ulang dari data latih saja; identitas *tokenizer* terhadap Tabel 2.14 dan contoh subbab 2.5.2 naskah; pemotongan `truncation=True, max_length=L_max` beserta token khusus; dokumen yang terpotong dan panjangnya sebelum/sesudah; *padding* per *batch*.

**Package:** `math` (bawaan Python); `numpy`; `pandas`; `transformers.AutoTokenizer`; `huggingface_hub`; `lib.bantu`.

In [ ]:
# ══ Tahap 7 — Pemeriksaan akhir tokenisasi (tambahan; konfigurasi tidak diubah) ══
# Package: math (bawaan Python); numpy; pandas; transformers.AutoTokenizer; huggingface_hub (versi)
import math
import numpy as np, pandas as pd
import huggingface_hub
from transformers import AutoTokenizer
mulai()
masuk = PATH["hasil"] / "tahap07_tokenisasi.json"
tk, BAGIAN = baca_json(masuk), list(CFG["proporsi"])
data = {b: pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet") for b in BAGIAN}
tok, L = AutoTokenizer.from_pretrained(CFG["checkpoint"]), tk["L_max"]     # dari cache; tidak mengunduh ulang

# 1) L_max dihitung ulang dari panjang token data latih saja
pl = np.array(tk["panjang_latih"])
L_ulang = CFG["b"] * math.ceil(np.percentile(pl, CFG["q"]) / CFG["b"])
# 2) Identitas tokenizer: contoh Tabel 2.14 dan contoh subbab 2.5.2 naskah
T214 = {"aplikasinya": ["aplikasinya"], "mantapp": ["mantap", "##p"], "gabisa": ["gab", "##isa"],
        "tiba-tiba": ["tiba", "-", "tiba"], "bagus!": ["bagus", "!"]}
s252 = [len(tok.tokenize(w)) for w in "aplikasinya error bgt gabisa buka".split()]
# 3) Pemotongan seperti saat pelatihan (truncation=True, max_length=L_max) dan padding per batch
penuh = {b: tok(list(d["text_clean"]))["input_ids"] for b, d in data.items()}
potong = {b: tok(list(d["text_clean"]), truncation=True, max_length=L)["input_ids"] for b, d in data.items()}
kena = {b: [int(i) for i, a in zip(data[b]["review_id"], penuh[b]) if len(a) > L] for b in BAGIAN}
utuh = all(a == c for b in BAGIAN for a, c in zip(penuh[b], potong[b]) if len(a) <= L)
ujung = all(c[0] == tok.cls_token_id and c[-1] == tok.sep_token_id for b in BAGIAN for c in potong[b])
bt = tok(list(data["latih"]["text_clean"][:16]), truncation=True, max_length=L, padding=True, return_tensors="np")
nyata = [len(x) for x in potong["latih"][:16]]

judul("7", "Pemeriksaan akhir tokenisasi", masukan=[rel(masuk), "hasil/tahap06_{latih,validasi,uji}.parquet"])
mml = "tidak diatur" if tok.model_max_length > 1e6 else tok.model_max_length
print(f"Tokenizer {type(tok).__name__} · kosakata {fmt_id(tok.vocab_size)} · potong sisi {tok.truncation_side} · "
      f"padding sisi {tok.padding_side} · model_max_length bawaan {mml} · huggingface_hub {huggingface_hub.__version__}")
print(f"Token khusus: [CLS]={tok.cls_token_id} [SEP]={tok.sep_token_id} [PAD]={tok.pad_token_id} [UNK]={tok.unk_token_id}")
print(f"(1) n latih {fmt_id(len(pl))} · P{CFG['q']} = maks latih = {int(pl.max())} → L_max = {L_ulang} (JSON {L})")
print("(2) Tabel 2.14: " + "; ".join(f"{k} → {' '.join(tok.tokenize(k))}" for k in T214) + f" | 2.5.2: {s252}")
print(f"(3) {'Bagian':<9}{'Maks sebelum':>13}{'Maks sesudah':>13}  Terpotong")
for b in BAGIAN:
    print(f"    {b:<9}{max(map(len, penuh[b])):>13}{max(map(len, potong[b])):>13}  {kena[b] or '-'}")
for i in kena["uji"]:
    n = len(penuh["uji"][data["uji"]["review_id"].tolist().index(i)])
    print(f"    review_id {i}: {n} → {L} token ({n - L} subkata terakhir dibuang; {fmt_id(L / n * 100, 1)}% dipertahankan)")
print(f"(4) padding per batch (16 dokumen latih pertama): lebar {bt['input_ids'].shape[1]} = terpanjang {max(nyata)}")
ya = lambda x: "ya" if x else "tidak"
cek("L_max ulang · JSON · n latih", (L_ulang, L, len(pl)), (TARGET["7"]["L_max"],) * 2 + (TARGET["6.1"]["latih"][0],))
cek("Tabel 2.14 cocok (kata)", sum(tok.tokenize(k) == v for k, v in T214.items()), len(T214))
cek("subbab 2.5.2: subkata per kata", s252, [1, 1, 1, 2, 1])
cek("terpotong latih · validasi · uji", [len(kena[b]) for b in BAGIAN], list(TARGET["7"]["terpotong"].values()))
cek("review_id terpotong", sum(kena.values(), []), TARGET["7"]["terpotong_id"])
cek("panjang sesudah potong <= L_max", ya(all(len(c) <= L for b in BAGIAN for c in potong[b])), "ya")
cek("tak terpotong identik · diawali [CLS], diakhiri [SEP]", [ya(utuh), ya(ujung)], ["ya", "ya"])
cek("attention_mask = panjang asli", ya(bt["attention_mask"].sum(1).tolist() == nyata), "ya")
selesai()

## Tahap 8.1 — Tulis `lib/wondr_latih.py` dan uji cepat

**Dasar naskah:** Bab II subbab 2.6.2 dan 2.8; pers. (2.14), (2.15), (2.18)–(2.22); Tabel 3.1. Keputusan teknis KT-5 (*weight decay* tidak dikenakan pada bias dan `LayerNorm.weight`), KT-6 (kepala klasifikasi bawaan dengan *pooler* hasil prapelatihan), KT-7 (konfigurasi lewat `pakai_konfig(CFG)`, *batch* evaluasi = B, FP32, `num_workers=0`, `fitness()` memakai `label_akhir`, uji determinisme) disetujui 21-10-2026.

**Package:** `gc`, `json`, `math`, `time`, `pathlib`, `importlib` (bawaan Python); `numpy`; `pandas`; `torch`; `scikit-learn` (`f1_score`); `transformers` (`AutoModelForSequenceClassification`, `AutoTokenizer`, `get_linear_schedule_with_warmup`); `lib.bantu` (`atur_seed`).

Sel 1/2 menulis `lib/wondr_latih.py` (fungsi saja). Sel 2/2 menguji: T dan T_w, parameter baru, dan pelatihan kecil (64 dokumen, 1 *epoch*) dua kali dengan *seed* sama. Hasil uji tidak disimpan.

In [ ]:
%%writefile lib/wondr_latih.py
# lib/wondr_latih.py -- fine-tuning IndoBERT (Tahap 8.1; KT-5, KT-6, KT-7 disetujui 21-10-2026).
# Isinya fungsi saja. Konfigurasi diambil dari CFG lewat pakai_konfig(CFG) di setiap sel pelatihan.
import gc, json, math, time
from pathlib import Path
import numpy as np, pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from lib.bantu import atur_seed

_AKAR = Path(__file__).resolve().parents[1]          # folder kerja .../skripsi_wondr_v3
_K = {}                                              # konfigurasi aktif (salinan CFG + L_max)
_CACHE = {}                                          # tokenizer dan input_ids per review_id, per sesi
TANPA_WD = ("bias", "LayerNorm.weight")              # KT-5: tanpa weight decay, seperti BERT asli


def pakai_konfig(cfg):
    """Salin CFG dan ambil L_max dari hasil Tahap 7; cache tokenisasi dikosongkan bila L_max/checkpoint berubah."""
    L = json.loads((_AKAR / "hasil" / "tahap07_tokenisasi.json").read_text(encoding="utf-8"))["L_max"]
    if (_K.get("checkpoint"), _K.get("L_max")) != (cfg["checkpoint"], L):
        _CACHE.clear()
    _K.clear(); _K.update(cfg); _K["L_max"] = L
    return L


def jadwal(n_latih, wr):
    """Pers. (2.21)-(2.22): T = ceil(n_latih / B) * E dan T_w = floor(wr * T)."""
    T = math.ceil(n_latih / _K["B"]) * _K["E"]
    return T, math.floor(wr * T)


def _tok():
    if "tok" not in _CACHE:
        _CACHE["tok"] = AutoTokenizer.from_pretrained(_K["checkpoint"])
    return _CACHE["tok"]


def _kode(df):
    """input_ids per dokumen dengan truncation=True, max_length=L_max; dihitung sekali per review_id."""
    ids = _CACHE.setdefault("ids", {})
    baru = [(r, t) for r, t in zip(df["review_id"], df["text_clean"]) if r not in ids]
    if baru:
        hasil = _tok()([t for _, t in baru], truncation=True, max_length=_K["L_max"])["input_ids"]
        ids.update({r: x for (r, _), x in zip(baru, hasil)})
    return [ids[r] for r in df["review_id"]]


def _muat(ids, y, acak, seed=None):
    """DataLoader dengan padding dinamis per batch (kanan, token [PAD]); batch = B; num_workers = 0."""
    pad = _tok().pad_token_id

    def kumpul(batch):
        n = max(len(a) for a, _ in batch)
        x = torch.full((len(batch), n), pad, dtype=torch.long)
        m = torch.zeros((len(batch), n), dtype=torch.long)
        for i, (a, _) in enumerate(batch):
            x[i, :len(a)] = torch.tensor(a)
            m[i, :len(a)] = 1
        return x, m, torch.tensor([b for _, b in batch], dtype=torch.long)

    g = torch.Generator().manual_seed(seed) if acak else None
    return DataLoader(list(zip(ids, y)), batch_size=_K["B"], shuffle=acak, generator=g, collate_fn=kumpul, num_workers=0)


@torch.no_grad()
def _peluang(model, ids, dev):
    """Peluang kelas Positif (softmax) per dokumen, urutan tetap."""
    model.eval()
    p = [torch.softmax(model(input_ids=x.to(dev), attention_mask=m.to(dev)).logits, dim=-1)[:, 1].cpu()
         for x, m, _ in _muat(ids, [0] * len(ids), acak=False)]
    return torch.cat(p).double().numpy()


def tebak(p1):
    """argmax dua kelas dari peluang Positif; seri (0,5) -> kelas 0."""
    return (np.asarray(p1) > 0.5).astype(int)


def parameter_baru():
    """Parameter yang tidak ada di checkpoint saat model klasifikasi dibuat (KT-6: harus hanya lapisan klasifikasi)."""
    _, info = AutoModelForSequenceClassification.from_pretrained(_K["checkpoint"], num_labels=2, output_loading_info=True)
    return sorted(info["missing_keys"]), sorted(info["unexpected_keys"])


def latih(df_latih, lr, wr, wd, seed, df_val=None, early_stopping=True, prediksi=None, maks_epoch=None):
    """Satu kali fine-tuning (Bab II 2.8). maks_epoch hanya untuk uji cepat; bawaan = E."""
    if not _K:
        raise RuntimeError("Panggil wl.pakai_konfig(CFG) lebih dulu.")
    if early_stopping and df_val is None:
        raise ValueError("early_stopping=True memerlukan df_val.")
    t0, dev = time.perf_counter(), torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ids_latih, prediksi = _kode(df_latih), prediksi or {}
    ids_val = _kode(df_val) if df_val is not None else None
    ids_pred = {k: _kode(d) for k, d in prediksi.items()}

    atur_seed(seed)                                  # sebelum model dibuat: bobot awal W_c, b_c bergantung seed
    model = AutoModelForSequenceClassification.from_pretrained(_K["checkpoint"], num_labels=2).to(dev)
    grup = [{"params": [p for n, p in model.named_parameters() if not n.endswith(TANPA_WD)], "weight_decay": wd},
            {"params": [p for n, p in model.named_parameters() if n.endswith(TANPA_WD)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(grup, lr=lr, betas=tuple(_K["adam_betas"]), eps=_K["adam_eps"])     # pers. (2.18)
    T, T_w = jadwal(len(df_latih), wr)
    sch = get_linear_schedule_with_warmup(opt, T_w, T)                  # pers. (2.20); T tetap walau berhenti dini
    muat = _muat(ids_latih, df_latih["label"].astype(int).tolist(), acak=True, seed=seed)

    f1_epoch, terbaik, ep_terbaik, tanpa_naik, prob = [], -1.0, None, 0, {}
    for ep in range(1, (maks_epoch or _K["E"]) + 1):
        model.train()
        for x, m, y in muat:
            loss = model(input_ids=x.to(dev), attention_mask=m.to(dev), labels=y.to(dev)).loss   # (2.15), tanpa bobot kelas
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), _K["v"])                         # pers. (2.19)
            opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        if early_stopping:
            f1 = float(f1_score(df_val["label"].astype(int), tebak(_peluang(model, ids_val, dev)),
                                average="macro", zero_division=0))
            f1_epoch.append(f1)
            if f1 > terbaik:                         # diperbarui hanya bila lebih besar
                terbaik, ep_terbaik, tanpa_naik = f1, ep, 0
                prob = {k: _peluang(model, i, dev) for k, i in ids_pred.items()}   # peluang di epoch terbaik
            else:
                tanpa_naik += 1
                if tanpa_naik >= _K["p"]:
                    break
    if not early_stopping:                           # CL: peluang dari bobot epoch terakhir
        prob = {k: _peluang(model, i, dev) for k, i in ids_pred.items()}

    del model, opt, sch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"f1_terbaik": terbaik if early_stopping else None, "epoch_terbaik": ep_terbaik, "f1_per_epoch": f1_epoch,
            "detik": time.perf_counter() - t0, "prob": prob, "T": T, "T_w": T_w, "perangkat": dev.type}


def fitness(lr, wr, wd):
    """Pers. (2.25): rata-rata f1_terbaik seed 42 dan 43; data Tahap 9.3, label = label_akhir (KT-7)."""
    baca = lambda b: pd.read_parquet(_AKAR / "hasil" / f"tahap09_{b}.parquet").assign(label=lambda d: d["label_akhir"])
    df_latih, df_val = baca("latih"), baca("validasi")
    hasil = {s: latih(df_latih, lr, wr, wd, s, df_val=df_val) for s in _K["seed_fitness"]}
    skor = [hasil[s]["f1_terbaik"] for s in _K["seed_fitness"]]
    return {"fitness": float(np.mean(skor)), "skor": skor, "hasil": hasil}

In [ ]:
# ══ Tahap 8.1 — Uji cepat lib/wondr_latih.py (sel 2/2; hasil tidak disimpan) ══
# Package: importlib (bawaan Python); numpy; pandas; torch; lib.wondr_latih (uji memori GPU: sel tambahan)
import importlib
import numpy as np, pandas as pd, torch
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
T = TARGET["8.1"]
L_max = wl.pakai_konfig(CFG)

# 1) Jadwal untuk n = 3.051 dokumen latih dan skema default (pers. 2.21-2.22)
T_, Tw_ = wl.jadwal(TARGET["6.1"]["latih"][0], CFG["default"]["wr"])
# 2) Parameter baru saat model klasifikasi dibuat (KT-6: hanya lapisan klasifikasi W_c, b_c)
baru, sisa = wl.parameter_baru()
# 3) Pelatihan kecil: 64 dokumen latih + 64 validasi, 1 epoch, skema default, dua kali dengan seed yang sama
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet").head(64)
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet").head(64)
d, s = CFG["default"], CFG["seed_fitness"][0]
r = [wl.latih(lat, d["lr"], d["wr"], d["wd"], s, df_val=val, prediksi={"val": val}, maks_epoch=1) for _ in range(2)]

judul("8.1", "Uji cepat lib/wondr_latih.py", keluaran="(tidak ada; hasil uji tidak disimpan)",
      masukan=["lib/wondr_latih.py", "64 dokumen tahap06_latih + 64 tahap06_validasi"])
print(f"L_max = {L_max} · jadwal n = {fmt_id(TARGET['6.1']['latih'][0])}: T = {T_} · T_w = {Tw_}")
print(f"Parameter baru  : {', '.join(baru)} ({len(baru)} tensor)")
print(f"Tidak terpakai  : {len(sisa)} tensor checkpoint (kepala prapelatihan)")
for i, x in enumerate(r, 1):
    print(f"Pelatihan {i}: lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · seed {s} · epoch terbaik "
          f"{x['epoch_terbaik']} · Macro F1 val {fmt_id(x['f1_terbaik'])} · {fmt_id(x['detik'], 1)} detik · {x['perangkat']}")
cek("T · T_w (n = 3.051, rho = 0,1)", (T_, Tw_), (T["T"], T["T_w"]))
cek("parameter baru hanya lapisan klasifikasi", baru, ["classifier.bias", "classifier.weight"])
ya, p = (lambda x: "ya" if x else "tidak"), r[0]["prob"]["val"]
cek("dua pelatihan identik (F1 · peluang)", [ya(r[0]["f1_terbaik"] == r[1]["f1_terbaik"]),
    ya(np.array_equal(p, r[1]["prob"]["val"]))], ["ya", "ya"])
cek("peluang di [0, 1] untuk 64 dokumen", ya(len(p) == 64 and ((p >= 0) & (p <= 1)).all()), "ya")
cek("pelatihan di GPU", r[0]["perangkat"], "cuda")
selesai()

### Tahap 8.1 — Uji memori GPU (tambahan, diminta pengguna)

Model, parameter pelatihan, dan *seed* tidak diubah. Mengukur empat metrik di empat titik (awal setelah ruang kerja dilepas, setelah pelatihan 1, setelah pelatihan 2, setelah ruang kerja cuBLAS dilepas): `torch.cuda.memory_allocated()` (blok *allocator* yang sedang dipakai, termasuk ruang kerja cuBLAS), `torch.cuda.memory_reserved()` (blok yang ditahan *allocator*, termasuk *cache* kosong), jumlah tensor CUDA yang masih hidup di Python, dan memori terpakai menurut *driver* (`torch.cuda.mem_get_info()`, termasuk konteks CUDA). Ruang kerja cuBLAS dilepas dengan `torch._C._cuda_clearCublasWorkspaces()` lalu `torch.cuda.empty_cache()`.

**Package:** `gc`, `os` (bawaan Python); `pandas`; `torch`; `lib.wondr_latih`; `lib.bantu`.

In [ ]:
# ══ Tahap 8.1 — Uji memori GPU (tambahan; model, parameter, dan seed tidak diubah) ══
# Package: gc, os (bawaan Python); pandas; torch; lib.wondr_latih
import gc, os
import pandas as pd, torch
import lib.wondr_latih as wl
mulai()
wl.pakai_konfig(CFG)
MiB = 2**20

def ukur():
    """[teralokasi, dicadangkan, tensor CUDA hidup di Python, terpakai di GPU menurut driver]."""
    gc.collect()
    hidup = sum(1 for o in gc.get_objects() if torch.is_tensor(o) and o.is_cuda)
    bebas, total = torch.cuda.mem_get_info()
    return [torch.cuda.memory_allocated() / MiB, torch.cuda.memory_reserved() / MiB, hidup, (total - bebas) / MiB]

def lepas():                                       # ruang kerja cuBLAS -> allocator, lalu cache allocator -> driver
    torch._C._cuda_clearCublasWorkspaces()
    torch.cuda.empty_cache()

# 1) Ukur di 4 titik dengan pelatihan kecil yang sama seperti sel 2/2 (64 dokumen, 1 epoch, skema default)
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet").head(64)
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet").head(64)
d, s = CFG["default"], CFG["seed_fitness"][0]
lepas(); M = {"awal (setelah dilepas)": ukur()}
for i in (1, 2):
    wl.latih(lat, d["lr"], d["wr"], d["wd"], s, df_val=val, prediksi={"val": val}, maks_epoch=1)
    M[f"setelah pelatihan {i}"] = ukur()
lepas(); M["setelah cuBLAS dilepas"] = ukur()
uk, jml = map(int, os.environ["CUBLAS_WORKSPACE_CONFIG"].strip(":").split(":"))
ws = 2 * uk * jml / 1024 + 1                        # Hitungan: 2 handle x (4.096 KiB x 8) + 1 MiB cuBLASLt (bawaan)

# 2) Ringkasan + cek
judul("8.1", "Uji memori GPU (tambahan)", masukan="lib/wondr_latih.py · 2 pelatihan kecil", keluaran="(tidak ada)")
print(f"{'Titik ukur':<24}{'Teralokasi':>11}{'Dicadangkan':>12}{'Tensor CUDA':>12}{'Terpakai GPU':>13}")
for k, (a, r, n, g) in M.items():
    print(f"{k:<24}{fmt_id(a, 1):>11}{fmt_id(r, 1):>12}{n:>12}{fmt_id(g, 0):>13}")
print("(MB; teralokasi/dicadangkan = allocator PyTorch; terpakai GPU = driver, termasuk konteks CUDA)")
v = list(M.values())
cek("tensor CUDA hidup setelah pelatihan 1 · 2", [v[1][2], v[2][2]], [0, 0])
cek("teralokasi setelah pelatihan 1 · 2 = ruang kerja cuBLAS (MB)", [v[1][0], v[2][0]], [ws, ws], toleransi=0.5)
cek("perubahan pelatihan 1 → 2: teralokasi · dicadangkan · terpakai GPU (MB)",
    [v[2][j] - v[1][j] for j in (0, 1, 3)], [0, 0, 0], toleransi=1)
cek("teralokasi · dicadangkan, awal dan setelah dilepas (MB, nilai aktual)",
    [v[0][0], v[0][1], v[3][0], v[3][1]], [0, 0, 0, 0], toleransi=0.5)
cek("penurunan terpakai GPU saat dilepas = cadangan yang dilepas (MB)", v[2][3] - v[3][3], v[2][1] - v[3][1], toleransi=2)
selesai()

## Tahap 8.2 — *Confident learning*: pelatihan 5 lipatan

**Dasar naskah:** Bab III tahap 8; Bab II subbab 2.3.4, pers. (2.6); Northcutt dkk. (2021). Lipatan: `StratifiedGroupKFold(5, shuffle=True, random_state=42)` dengan `groups=group_key` dan `y=label` (label awal); setiap model memakai skema *default* (lr 2e-5, *warmup ratio* 0,1, *weight decay* 0,01), *seed* 42, 4 *epoch* tanpa *early stopping*. Data uji tidak dibaca.

**Package:** `os`, `importlib` (bawaan Python); `numpy`; `pandas`; `scikit-learn` (`StratifiedGroupKFold`, `f1_score`); `lib.wondr_latih`.

Sel 1/2 melatih 5 model dan menyimpan peluang Positif untuk lipatan yang ditahan (di luar sampel) dan untuk seluruh data validasi ke `log/cl_lipatan_k.parquet`. Lipatan yang sudah tersimpan dilewati, sehingga sel bisa dijalankan ulang bila Colab terputus. Sel 2/2 membaca kelima berkas, menyusun ulang lipatan, lalu memeriksa kelengkapan peluang (berdiri sendiri, tanpa GPU).

In [ ]:
# ══ Tahap 8.2 — Confident learning: pelatihan 5 lipatan (sel 1/2) ══
# Package: os, importlib (bawaan Python); numpy; pandas; scikit-learn (StratifiedGroupKFold, f1_score); lib.wondr_latih
import os, importlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import lib.wondr_latih as wl
importlib.reload(wl)                                        # pakai isi lib/wondr_latih.py yang terbaru
mulai()
wl.pakai_konfig(CFG)
d, K, s = CFG["default"], CFG["cl_lipatan"], CFG["cl_seed_latih"]

# 1) Baca data latih dan validasi (label awal); data uji tidak dibaca
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet")
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet")

# 2) Lipatan berbasis grup (pers. 2.6), terstratifikasi label awal
lipatan = np.empty(len(lat), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=CFG["cl_seed_lipatan"])
for k, (_, idx) in enumerate(sgkf.split(lat, lat["label"], groups=lat["group_key"])):
    lipatan[idx] = k

# 3) Setiap lipatan k: latih pada 4 lipatan lain (skema default, E epoch, tanpa early stopping), lalu prediksi
#    lipatan k dan seluruh data validasi. Satu berkas per lipatan; lipatan yang sudah tersimpan dilewati.
for k in range(K):
    berkas = PATH["log"] / f"cl_lipatan_{k}.parquet"
    tr, te = lat[lipatan != k], lat[lipatan == k]
    if berkas.exists():
        print(f"Lipatan {k}: sudah tersimpan di {rel(berkas)}, dilewati")
        continue
    r = wl.latih(tr, d["lr"], d["wr"], d["wd"], s, early_stopping=False, prediksi={"latih": te, "validasi": val})
    out = pd.concat([pd.DataFrame({"review_id": x["review_id"].to_numpy(), "bagian": b, "p1": r["prob"][b]})
                     for b, x in (("latih", te), ("validasi", val))], ignore_index=True)
    out.assign(lipatan=k, n_latih=len(tr), detik=r["detik"]).to_parquet(berkas.with_suffix(".tmp"), index=False)
    os.replace(berkas.with_suffix(".tmp"), berkas)            # tulis atomik: aman bila Colab terputus
    f1 = f1_score(te["label"], wl.tebak(r["prob"]["latih"]), average="macro", zero_division=0)
    print(f"Lipatan {k}: lr {d['lr']} · wr {fmt_id(d['wr'], 2)} · wd {fmt_id(d['wd'], 2)} · seed {s} · {CFG['E']} epoch · "
          f"latih {fmt_id(len(tr))} · prediksi {fmt_id(len(te))} + {fmt_id(len(val))} · Macro F1 lipatan {fmt_id(f1)} · "
          f"{fmt_id(r['detik'] / 60, 1)} menit")
selesai()

In [ ]:
# ══ Tahap 8.2 — Confident learning: ringkasan dan cek 5 lipatan (sel 2/2) ══
# Package: numpy; pandas; scikit-learn (StratifiedGroupKFold, f1_score); lib.wondr_latih (tebak)
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
import lib.wondr_latih as wl
mulai()
K = CFG["cl_lipatan"]

# 1) Baca data latih/validasi dan kelima berkas lipatan; lipatan disusun ulang untuk mencocokkan berkas
lat = pd.read_parquet(PATH["hasil"] / "tahap06_latih.parquet")
val = pd.read_parquet(PATH["hasil"] / "tahap06_validasi.parquet")
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)], ignore_index=True)
lipatan = np.empty(len(lat), dtype=int)
sgkf = StratifiedGroupKFold(n_splits=K, shuffle=True, random_state=CFG["cl_seed_lipatan"])
for k, (_, idx) in enumerate(sgkf.split(lat, lat["label"], groups=lat["group_key"])):
    lipatan[idx] = k
oof = cl[cl.bagian == "latih"].merge(lat[["review_id", "label", "group_key"]], on="review_id", how="left")
f1 = lambda g: f1_score(g["label"], wl.tebak(g["p1"]), average="macro", zero_division=0)

# 2) Ringkasan per lipatan (Macro F1 terhadap label awal, deskriptif) + cek
judul("8.2", "Confident learning: 5 lipatan", masukan="hasil/tahap06_latih.parquet, hasil/tahap06_validasi.parquet",
      keluaran=f"log/cl_lipatan_0..{K - 1}.parquet ({fmt_id(len(cl))} baris)")
print(f"{'k':<3}{'n latih':>9}{'n lipatan':>11}{'n validasi':>12}{'Non-Positif':>13}{'Macro F1':>10}{'Durasi':>11}")
for k, g in oof.groupby("lipatan"):
    m = cl[cl.lipatan == k]
    print(f"{k:<3}{fmt_id(int(m.n_latih.iloc[0])):>9}{fmt_id(len(g)):>11}{fmt_id(int((m.bagian == 'validasi').sum())):>12}"
          f"{fmt_id((g.label == 0).mean() * 100, 1) + '%':>13}{fmt_id(f1(g)):>10}{fmt_id(m.detik.iloc[0] / 60, 1) + ' mnt':>11}")
garis()
print(f"Macro F1 di luar sampel, seluruh data latih (label awal, deskriptif): {fmt_id(f1(oof))} · "
      f"total durasi {fmt_id(cl.groupby('lipatan').detik.first().sum() / 60, 1)} menit")
ya = lambda x: "ya" if x else "tidak"
cek("dokumen latih dengan tepat 1 peluang di luar sampel", [int(oof.review_id.nunique()), ya(oof.review_id.is_unique)],
    [len(lat), "ya"])
cek("lipatan di berkas = lipatan yang disusun ulang", ya(oof[["review_id", "lipatan"]].merge(pd.DataFrame(
    {"review_id": lat.review_id, "k": lipatan}), on="review_id", how="outer").eval("lipatan == k").all()), "ya")
cek("group_key terbelah antarlipatan", int((oof.groupby("group_key").lipatan.nunique() > 1).sum()), 0)
cek("dokumen validasi dengan tepat 5 peluang", int((cl[cl.bagian == "validasi"].groupby("review_id").lipatan.nunique()
    == K).sum()), len(val))
cek("peluang di [0, 1] · tanpa NaN", [ya(cl.p1.between(0, 1).all()), ya(cl.p1.notna().all())], ["ya", "ya"])
cek("review_id di luar latih/validasi (data uji tidak disentuh)",
    len(set(cl.review_id) - set(lat.review_id) - set(val.review_id)), 0)
selesai()

## Tahap 8.3 — Ambang, *confident joint*, dan kandidat

**Dasar naskah:** Bab III tahap 8; Bab II subbab 2.3.4, pers. (2.3)–(2.4), Tabel 2.9; Northcutt dkk. (2021). Rumus ditulis sendiri; pustaka `cleanlab` tidak dipakai. Ambang dan *confident joint* dihitung terpisah untuk data latih (peluang di luar sampel) dan data validasi (rata-rata peluang 5 model). Data uji tidak dibaca.

**Package:** `math`, `importlib` (bawaan Python); `numpy`; `pandas`; `lib.wondr_label` (`confident_learning`, `per_bagian`).

Sel 1/2 menulis `lib/wondr_label.py` (fungsi saja). Sel 2/2 menguji fungsi dengan contoh Tabel 2.9, menerapkannya pada latih dan validasi, lalu menyimpan `hasil/tahap08_kandidat.parquet` dan `bab4/tabel/tahap08_cj.csv`. Dokumen yang tidak memenuhi ambang kelas mana pun tidak dihitung (`kelas_taksiran` kosong) dan bukan kandidat.

In [ ]:
%%writefile lib/wondr_label.py
# lib/wondr_label.py -- confident learning dua kelas (Tahap 8.3). Rumus ditulis sendiri sesuai pers. (2.3)-(2.4),
# Bab II subbab 2.3.4; pustaka cleanlab tidak dipakai. Isinya fungsi saja.
import math
import numpy as np, pandas as pd


def confident_learning(p1, y):
    """Ambang kelas, kelas taksiran, confident joint, dan kandidat label salah; p0 = 1 - p1.

    t_j      = rata-rata p_j pada dokumen berlabel j                                   (pers. 2.3)
    taksiran = argmax p_l di antara kelas l dengan p_l >= t_l; -1 bila tidak ada yang memenuhi ambang
               (dokumen itu tidak dihitung); bila p0 = p1 dan keduanya lolos, dipilih kelas 0
    C[i, j]  = banyak dokumen berlabel i dengan kelas taksiran j; baris = label, kolom = taksiran (pers. 2.4)
    kandidat = dokumen yang dihitung dengan kelas taksiran != label (unsur di luar diagonal C)
    """
    p1, y = np.asarray(p1, dtype=float), np.asarray(y)
    if not (set(np.unique(y)) == {0, 1} and np.all((p1 >= 0) & (p1 <= 1))):
        raise ValueError("y harus memuat kelas 0 dan 1 saja, dan p1 harus di [0, 1] tanpa NaN.")
    y = y.astype(int)
    P = np.column_stack([1.0 - p1, p1])                       # kolom j = peluang kelas j
    t = np.array([math.fsum(P[y == j, j]) / (y == j).sum() for j in (0, 1)])   # pers. (2.3); jumlah dibulatkan tepat
    lolos = P >= t
    taksiran = np.where(lolos.any(axis=1), np.where(lolos, P, -np.inf).argmax(axis=1), -1)
    dihitung = taksiran >= 0
    C = np.zeros((2, 2), dtype=int)
    np.add.at(C, (y[dihitung], taksiran[dihitung]), 1)        # pers. (2.4)
    return {"t": t, "taksiran": taksiran, "dihitung": dihitung, "C": C, "kandidat": dihitung & (taksiran != y)}


def per_bagian(d):
    """confident_learning() terpisah untuk setiap bagian (Bab III tahap 8); d berkolom review_id, bagian, label, p1.

    Mengembalikan (tabel per dokumen: review_id, bagian, label, p1, kelas_taksiran [<NA> = tidak dihitung], kandidat;
    tabel per bagian: n, t0, t1, tidak_dihitung, C00, C01, C10, C11, kandidat, kandidat_pct)."""
    dok, cj = [], []
    for b, x in d.groupby("bagian", sort=False):
        r = confident_learning(x["p1"], x["label"])
        taksiran = pd.array(np.where(r["dihitung"], r["taksiran"], pd.NA), dtype="Int64")
        dok.append(x[["review_id", "bagian", "label", "p1"]].assign(kelas_taksiran=taksiran, kandidat=r["kandidat"]))
        cj.append({"bagian": b, "n": len(x), "t0": r["t"][0], "t1": r["t"][1], "tidak_dihitung": int((~r["dihitung"]).sum()),
                   **{f"C{i}{j}": int(r["C"][i, j]) for i in (0, 1) for j in (0, 1)}, "kandidat": int(r["kandidat"].sum())})
    cj = pd.DataFrame(cj)
    cj["kandidat_pct"] = 100 * cj["kandidat"] / cj["n"]
    return pd.concat(dok, ignore_index=True), cj

In [ ]:
# ══ Tahap 8.3 — Ambang, confident joint, dan kandidat (sel 2/2) ══
# Package: importlib (bawaan Python); numpy; pandas; lib.wondr_label (confident_learning, per_bagian)
import importlib
import numpy as np, pandas as pd
import lib.wondr_label as wlab
importlib.reload(wlab)                                      # pakai isi lib/wondr_label.py yang terbaru
mulai()
T, K, n_uji, ya = TARGET["8.3"], CFG["cl_lipatan"], TARGET["6.1"]["uji"][0], (lambda x: "ya" if x else "tidak")

# 1) Peluang Positif: latih = di luar sampel (1 model); validasi = rata-rata 5 model. Data uji tidak dibaca.
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)])
p = cl.groupby(["bagian", "review_id"], as_index=False).agg(p1=("p1", "mean"), n_model=("p1", "size"))
d = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "label"]).assign(bagian=b)
               for b in ("latih", "validasi")]).merge(p, on=["bagian", "review_id"], how="left")

# 2) Ambang (2.3), confident joint (2.4), dan kandidat, terpisah untuk latih dan validasi; lalu simpan
dok, cj = wlab.per_bagian(d)
dok.to_parquet(PATH["hasil"] / "tahap08_kandidat.parquet", index=False)
simpan_tabel(cj, "tahap08_cj")

# 3) Ringkasan dan cek; fungsi diuji dengan contoh Tabel 2.9 (d1-d6)
u = wlab.confident_learning([0.95, 0.90, 0.15, 0.10, 0.20, 0.85], [1, 1, 1, 0, 0, 0])
judul("8.3", "Ambang, confident joint, dan kandidat", masukan=f"log/cl_lipatan_0..{K - 1}.parquet, tahap06 latih/validasi",
      keluaran=[f"hasil/tahap08_kandidat.parquet ({fmt_id(len(dok))} baris)", "bab4/tabel/tahap08_cj.csv"])
print(f"{'Bagian':<10}{'n':>6}{'t0':>8}{'t1':>8}{'C00':>7}{'C01 NP→P':>10}{'C10 P→NP':>10}{'C11':>7}{'Kandidat':>15}"
      f"{'Tdk dihitung':>14}")
for c in cj.itertuples():
    print(f"{c.bagian:<10}{fmt_id(c.n):>6}{fmt_id(c.t0):>8}{fmt_id(c.t1):>8}{fmt_id(c.C00):>7}{fmt_id(c.C01):>10}"
          f"{fmt_id(c.C10):>10}{fmt_id(c.C11):>7}{fmt_id(c.kandidat) + ' (' + fmt_id(c.kandidat_pct, 1) + '%)':>15}"
          f"{fmt_id(c.tidak_dihitung):>14}")
print(f"C: baris = label awal, kolom = kelas taksiran · dokumen untuk lembar anotasi (Tahap 9.1): "
      f"{' + '.join(fmt_id(k) for k in cj.kandidat)} + uji {fmt_id(n_uji)} = {fmt_id(int(cj.kandidat.sum()) + n_uji)}")
cek("contoh Tabel 2.9: t0 · t1", u["t"].tolist(), [T["t0"], T["t1"]], toleransi=5e-5)
cek("contoh Tabel 2.9: C · kandidat (d)", [u["C"].tolist(), (np.flatnonzero(u["kandidat"]) + 1).tolist()], [T["C"], T["kandidat"]])
cek("peluang per dokumen: latih 1 model · validasi 5 model", [ya(d.query("bagian == 'latih'").n_model.eq(1).all()),
    ya(d.query("bagian == 'validasi'").n_model.eq(K).all())], ["ya", "ya"])
for c in cj.itertuples():
    cek(f"{c.bagian}: jumlah kandidat", c.kandidat, 0, cara=">")
    cek(f"{c.bagian}: kandidat (%)", c.kandidat_pct, 40, cara="<=", d=1)
    cek(f"{c.bagian}: diagonal C dominan (C00 > C01 · C11 > C10)", [ya(c.C00 > c.C01), ya(c.C11 > c.C10)], ["ya", "ya"])
selesai()

### Tahap 8.3 — Audit (tambahan, diminta pengguna)

Hanya membaca berkas; tidak ada yang ditulis atau diubah. Semua angka dihitung ulang secara mandiri tanpa `lib.wondr_label`. **Audit A** memeriksa sumber peluang (latih: 1 model di luar sampel; validasi: rata-rata 5 model), ambang pers. (2.3), kecocokan p₁ dan label dengan log CL dan tahap06, kolom berkas, serta data uji (hanya `review_id`: jumlah 656 dan tidak beririsan). **Audit B** memeriksa aturan ambang pers. (2.4): pita dokumen yang tidak dihitung (1 − t₀ < p₁ < t₁), kelas taksiran, C, kandidat = C₀₁ + C₁₀, penyebut persentase, dan perbandingan dengan argmax biasa.

**Package:** `numpy`; `pandas`.

In [ ]:
# ══ Tahap 8.3 — Audit A: sumber peluang, ambang, dan isi berkas (tambahan; hanya membaca berkas) ══
# Package: pandas  (dihitung ulang secara mandiri, tanpa lib.wondr_label)
import pandas as pd
mulai()
K, n_lat, ya = CFG["cl_lipatan"], TARGET["6.1"]["latih"][0], (lambda x: "ya" if x else "tidak")
e = lambda v: f"{v:.1e}".replace(".", ",")

# 1) Baca keluaran 8.3, log lipatan CL, label awal tahap06, dan review_id data uji (tanpa teks dan label)
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet")
cj = pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian")
cl = pd.concat([pd.read_parquet(PATH["log"] / f"cl_lipatan_{k}.parquet") for k in range(K)])
lab = pd.concat([pd.read_parquet(PATH["hasil"] / f"tahap06_{b}.parquet", columns=["review_id", "label"]).assign(bagian=b)
                 for b in ("latih", "validasi")])
uji = pd.read_parquet(PATH["hasil"] / "tahap06_uji.parquet", columns=["review_id"]).review_id
# 2) Peluang dari log: latih = 1 model (lipatan yang tidak melatihnya), validasi = rata-rata 5 model; lalu ambang (2.3)
sumber = cl.groupby(["bagian", "review_id"]).agg(p1=("p1", "mean"), n_model=("p1", "size")).reset_index()
u = (lab.merge(sumber, on=["bagian", "review_id"], how="left")
        .merge(kd.set_index(["bagian", "review_id"]).add_suffix("_b").reset_index(), on=["bagian", "review_id"],
               how="outer", indicator=True))
lip = cl[cl.bagian == "latih"].groupby("lipatan").agg(n_latih=("n_latih", "first"), n_pred=("review_id", "size"))
judul("8.3", "Audit A: sumber peluang dan ambang", masukan=["hasil/tahap08_kandidat.parquet + bab4/tabel/tahap08_cj.csv",
      f"log/cl_lipatan_0..{K - 1}.parquet + tahap06 (uji: review_id saja)"])
print("Kolom berkas 8.3:", " · ".join(f"{c} ({t})" for c, t in kd.dtypes.astype(str).items()))
print("Lipatan 0-4, n latih + n diprediksi:", " · ".join(f"{fmt_id(int(r.n_latih))} + {fmt_id(int(r.n_pred))}" for _, r in lip.iterrows()))
print(f"{'Bagian':<9}{'n':>6}{'model/dok':>10}{'t0 ulang':>10}{'t0 CSV':>9}{'t1 ulang':>10}{'t1 CSV':>9}{'|Δt| maks':>11}"
      f"{'|Δp1| maks':>12}")
R = []
for b, x in u.groupby("bagian", sort=False):
    c, y = cj.loc[b], x.label
    t0, t1 = (1 - x.p1[y == 0]).mean(), x.p1[y == 1].mean()          # pers. (2.3), p0 = 1 - p1
    dt, dp = max(abs(t0 - c.t0), abs(t1 - c.t1)), float((x.p1 - x.p1_b).abs().max())
    print(f"{b:<9}{fmt_id(len(x)):>6}{'/'.join(map(str, sorted(x.n_model.unique()))):>10}{fmt_id(t0):>10}{fmt_id(c.t0):>9}"
          f"{fmt_id(t1):>10}{fmt_id(c.t1):>9}{e(dt):>11}{e(dp):>12}")
    R.append((b, [int((x._merge == "both").sum()), int(x.p1.isna().sum()), ya((y == x.label_b).all()),
                  ya(x.n_model.eq(1 if b == "latih" else K).all()), ya(max(dt, dp) <= 1e-12)], [len(x), 0, "ya", "ya", "ya"]))
for b, aktual, target in R:
    cek(f"{b}: baris berpasangan · p1 NaN · label = tahap06 · model per dokumen · |Δt|, |Δp1| ≤ 1e-12", aktual, target)
cek("setiap lipatan: n latih + n diprediksi = n data latih", ya((lip.n_latih + lip.n_pred == n_lat).all()), "ya")
cek("kolom berkas = spesifikasi 8.3", list(kd.columns), ["review_id", "bagian", "label", "p1", "kelas_taksiran", "kandidat"])
cek("uji: n dari berkas · irisan dengan berkas 8.3 · irisan dengan log CL", [len(uji), len(set(uji) & set(kd.review_id)),
    len(set(uji) & set(cl.review_id))], [TARGET["6.1"]["uji"][0], 0, 0])
selesai()

In [ ]:
# ══ Tahap 8.3 — Audit B: aturan ambang, dokumen tidak dihitung, C, dan kandidat (tambahan; hanya membaca berkas) ══
# Package: numpy; pandas  (dihitung ulang secara mandiri, tanpa lib.wondr_label)
import numpy as np, pandas as pd
mulai()
ya = lambda x: "ya" if x else "tidak"

# 1) Baca keluaran 8.3 (p1 dan label sudah dicocokkan dengan log CL dan tahap06 di Audit A)
kd = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet")
cj = pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian")

# 2) Hitung ulang per bagian dengan aturan pers. (2.4): kelas j lolos bila p_j >= t_j; argmax hanya di antara yang lolos
judul("8.3", "Audit B: aturan ambang, C, dan kandidat", masukan="hasil/tahap08_kandidat.parquet + bab4/tabel/tahap08_cj.csv")
print(f"{'Bagian':<9}{'pita tidak dihitung':>24}{'p1 tdk dihitung':>17}{'tdk NP/P':>10}{'dua lolos':>10}{'salah argmax':>13}"
      f"{'kandidat':>9}{'⊆ argmax':>9}{'argmax di pita':>15}")
R = []
for b, x in kd.groupby("bagian", sort=False):
    p1, y, c = x.p1.to_numpy(), x.label.to_numpy(), cj.loc[b]
    t0, t1 = (1 - p1[y == 0]).mean(), p1[y == 1].mean()
    l0, l1 = (1 - p1) >= t0, p1 >= t1
    tak = np.where(l0 & l1, (p1 > 1 - p1).astype(int), np.where(l1, 1, np.where(l0, 0, -1)))
    kand, salah, tdk = (tak >= 0) & (tak != y), (p1 > 0.5).astype(int) != y, tak < 0     # salah = argmax biasa != label
    C = [[int(((y == i) & (tak == j)).sum()) for j in (0, 1)] for i in (0, 1)]
    print(f"{b:<9}{fmt_id(1 - t0) + ' < p1 < ' + fmt_id(t1):>24}{fmt_id(p1[tdk].min()) + '–' + fmt_id(p1[tdk].max()):>17}"
          f"{fmt_id(int((tdk & (y == 0)).sum())) + '/' + fmt_id(int((tdk & (y == 1)).sum())):>10}{fmt_id(int((l0 & l1).sum())):>10}"
          f"{fmt_id(int(salah.sum())):>13}{fmt_id(int(kand.sum())):>9}{ya(not (kand & ~salah).any()):>9}"
          f"{fmt_id(int((salah & tdk).sum())):>15}")
    R.append((b, x, c, tak, kand, C, int(((p1 > 1 - t0) & (p1 < t1)).sum())))
for b, x, c, tak, kand, C, pita in R:
    cek(f"{b}: tidak dihitung = dokumen di pita · kelas_taksiran kosong", [int((tak < 0).sum()), int(x.kelas_taksiran.isna()
        .sum())], [pita, int(c.tidak_dihitung)])
    cek(f"{b}: taksiran · C · kandidat = berkas/CSV", [ya((x.kelas_taksiran.fillna(-1).to_numpy() == tak).all()),
        ya(C == [[c.C00, c.C01], [c.C10, c.C11]]), ya((x.kandidat.to_numpy() == kand).all())], ["ya"] * 3)
    cek(f"{b}: kandidat = C01 + C10 · % kandidat (penyebut n = {fmt_id(len(x))})", [int(kand.sum()),
        round(100 * kand.sum() / len(x), 4)], [int(c.C01 + c.C10), round(c.kandidat_pct, 4)])
selesai()

### Tahap 8.3 — Audit C: kelas taksiran, pita tidak dihitung, dan batas ambang (tambahan, diminta pengguna)

Hanya membaca berkas. Menampilkan tabel silang status CL (kandidat / dihitung dan sesuai label / tidak dihitung) × argmax biasa (p₁ > 0,5) terhadap label awal, contoh dokumen yang salah menurut argmax tetapi berada di pita tidak dihitung (1 − t₀ < p₁ < t₁), dokumen terdekat ke kedua batas ambang, dan uji kecil bahwa p tepat sama dengan ambang ikut dihitung (batas inklusif ≥, pers. 2.4).

**Package:** `numpy`; `pandas`; `lib.wondr_label` (`confident_learning`, hanya untuk uji batas).

In [ ]:
# ══ Tahap 8.3 — Audit C: kelas taksiran, pita tidak dihitung, dan batas ambang (tambahan; hanya membaca berkas) ══
# Package: numpy; pandas; lib.wondr_label (confident_learning, hanya untuk uji batas)
import numpy as np, pandas as pd
import lib.wondr_label as wlab
mulai()
ya = lambda x: "ya" if x else "tidak"

# 1) Baca keluaran 8.3; ambang per bagian dari CSV; argmax biasa = tebakan p1 > 0,5 (KT-7)
x = pd.read_parquet(PATH["hasil"] / "tahap08_kandidat.parquet").join(
    pd.read_csv(PATH["tabel"] / "tahap08_cj.csv").set_index("bagian")[["t0", "t1"]], on="bagian")
x = x.assign(p0=1 - x.p1, argmax=(x.p1 > 0.5).astype(int), lolos0=(1 - x.p1) >= x.t0, lolos1=x.p1 >= x.t1)
x["status"] = np.select([x.kandidat, x.kelas_taksiran.notna()], ["kandidat CL", "dihitung, = label"], "tidak dihitung")
x["argmax_vs_label"] = np.where(x.argmax == x.label, "argmax benar", "argmax salah")

# 2) Status CL x argmax; contoh dokumen salah-argmax di pita (review_id terkecil per label) + 1 kandidat pembanding
judul("8.3", "Audit C: kelas taksiran dan pita", masukan="hasil/tahap08_kandidat.parquet + bab4/tabel/tahap08_cj.csv")
print(pd.crosstab([x.bagian, x.status], x.argmax_vs_label, margins=True, margins_name="total").map(fmt_id).to_string(), "\n")
pita = x[x.kelas_taksiran.isna() & (x.argmax != x.label)]
contoh = pd.concat([pita.sort_values("review_id").groupby(["bagian", "label"]).head(1),
                    x[x.kandidat].sort_values("review_id").groupby("bagian").head(1)]).sort_values(["bagian", "kandidat"])
kol = ["bagian", "review_id", "label", "p1", "p0", "t0", "t1", "lolos0", "lolos1", "argmax", "kelas_taksiran", "kandidat"]
print(contoh[kol].to_string(index=False, formatters={k: (lambda v: fmt_id(v)) for k in ("p1", "p0", "t0", "t1")}), "\n")
# 3) Dokumen terdekat ke batas ambang: selisih terkecil di kedua sisi; jumlah yang tepat sama dengan ambang
for b, g in x.groupby("bagian", sort=False):
    d1, d0 = g.p1 - g.t1, g.p0 - g.t0                         # >= 0 berarti lolos (batas inklusif)
    print(f"{b:<9} p1 − t1: terdekat di bawah {fmt_id(d1[d1 < 0].max(), 6)} · di atas/sama {fmt_id(d1[d1 >= 0].min(), 6)} · "
          f"tepat sama {int((d1 == 0).sum())} | p0 − t0: terdekat di bawah {fmt_id(d0[d0 < 0].max(), 6)} · "
          f"di atas/sama {fmt_id(d0[d0 >= 0].min(), 6)} · tepat sama {int((d0 == 0).sum())}")
u = wlab.confident_learning([0.75, 0.75, 0.25, 0.25], [1, 1, 0, 0])   # t0 = t1 = 0,75; semua p tepat = ambang
for b, g in x.groupby("bagian", sort=False):
    cek(f"{b}: kosong ⇔ tidak lolos t0 maupun t1 · kandidat ⊆ argmax salah · dihitung & = label ⇒ argmax benar",
        [ya((g.kelas_taksiran.isna() == ~(g.lolos0 | g.lolos1)).all()), ya((g.argmax[g.kandidat] != g.label[g.kandidat]).all()),
         ya((g.argmax == g.label)[g.status == "dihitung, = label"].all())], ["ya"] * 3)
cek("uji batas: p tepat = ambang ikut dihitung (inklusif ≥) · taksiran", [ya(u["dihitung"].all()), u["taksiran"].tolist()],
    ["ya", [1, 1, 0, 0]])
selesai()